In [1]:
from fastapi import APIRouter, Query, HTTPException
from typing import Optional, List, Dict, Any, Literal
from datetime import datetime, timedelta
from collections import defaultdict
import pandas as pd
import json
import re
import sys
import os
sys.path.insert(0, r"D:/A0_Project")  # 讓它優先被找到
from PI_SYSTEM.models.sql_db_connect import MySQLConnet

db = MySQLConnet('piaoi_ol_defect_map')
cim_db = MySQLConnet('cim_piaoi')
old_db = MySQLConnet('l6a01_project')

#db = MySQLConnet('piaoi_inspection_density')
#from PI_SYSTEM.models.density.cim_density_job import Config as DensityJobConfig
#from PI_SYSTEM.models.density.API_Config import API_Config

In [20]:
na_tools = []
for file in os.listdir('PI_SYSTEM\models\inspection_density\output_csv'):
    df = pd.read_csv('D:\A0_Project\PI_SYSTEM\models\inspection_density\output_csv//' + file)
    na_tools = [v for v in set(na_tools + df['TOOL_ID'].unique().tolist())]
na_tools

['CAPIC607', 'CAPIC107', 'CAPIC507', 'CAPIC407', 'CAPIC207']

In [2]:
bpi_same_db = MySQLConnet('piaoi_bpi_same_point')
inspec_db = MySQLConnet('piaoi_inspection_density')
bpi_same_tbn = 'bpi_same_point_202606'
cim_tbn = 'cim_pi_glass_202606'

inspec_df = inspec_db.get_table('inspection_api_summary_202606')
print(inspec_df.iloc[0,:].to_dict())
bpi_same_df = bpi_same_db.get_table(bpi_same_tbn)
bpi_same_df.head()

{'pi_hour': Timestamp('2026-06-01 07:00:00'), 'shift_day': Timestamp('2026-06-01 00:00:00'), 'shift_week': '2026W23', 'shift_month': '202606', 'shift_start': Timestamp('2026-06-01 07:30:00'), 'shift_end': Timestamp('2026-06-01 08:30:00'), 'line_id': 'CAPIC207', 'model': 'G215HVN01', 'glass_type': 'CF', 'maingroup_glass_count': 44, 'maingroup_defect_count': 314, 'maingroup_density': 7.136363636363637, 'defect_code_glass_count': 44, 'small_defect_count': 212, 'middle_defect_count': 71, 'large_defect_count': 24, 'over_defect_count': 7, 'glass': 'AL5H151YSP,AL5H151YSQ,AL5H151YSJ,AL5H151YY5,AL5H151YXY,AL5H151Z1R,AL5H151Z1P,AL5H151Z1N,AL5H151Z37,AL5H151Z1M,AL5H151YTN,AL5H151YVD,AL5H151YV6,AL5H151YV5,AL5H151YVB,AL5H151YZJ,AL5H151YXT,AL5H151YZH,AL5H151YY1,AL5H151YTA,AL5H151YZC,AL5H151YZK,AL5H151YZN,AL5H151YZM,AL5H151YRK,AL5H151YQW,AL5H151YQ2,AL5H151YW4,AL5H151YZP,AL5H151YT9,AL5H151YZD,AL5H151YWL,AL5H151YPA,AL5H151Z07,AL5H151YWK,AL5H151Z0G,AL5H151Z09,AL5H151Z0J,AL5H151YNK,AL5H151Z1J,AL5H151YYU,

,id,model,glass_side,glass_id,scan_hour,run_day,tab,bpi_aoi,bpi_line_id,bpi_recipe_id,...,api_over_defect_count,pair_status,pair_message,default_offset_um,matched_points_json,comment,action,editor,modify_time,gen_time
0,123,M315TAN01,TFT,3N6A5JE4B,2026-06-03 11:00:00,2026-06-03,PISpot,aoi200,CAPIC200,4363,...,0,OK,,20,"[{""offset_um"": 20, ""distance"": 17.691806012954...",,,,NaT,2026-06-23 10:07:40
1,124,M315TAN01,TFT,3N6A5JE4B,2026-06-03 10:00:00,2026-06-03,UPI,aoi200,CAPIC200,4363,...,6,OK,,20,"[{""offset_um"": 20, ""distance"": 0.0, ""dx"": 0.0,...",,,,NaT,2026-06-23 10:07:40
2,125,M315TAN01,TFT,3N6A5JE4C,2026-06-03 10:00:00,2026-06-03,PISpot,aoi200,CAPIC200,4363,...,0,OK,,20,[],,,,NaT,2026-06-23 10:07:40
3,126,M315TAN01,TFT,3N6A5JE4C,2026-06-03 10:00:00,2026-06-03,UPI,aoi200,CAPIC200,4363,...,0,OK,,20,"[{""offset_um"": 20, ""distance"": 1.0, ""dx"": 0.0,...",,,,NaT,2026-06-23 10:07:40
4,127,M340QAR1B,TFT,VP6G4N88J,2026-06-05 14:00:00,2026-06-05,PISpot,aoi200,CAPIC400,4298,...,1,OK,,20,"[{""offset_um"": 20, ""distance"": 4.0, ""dx"": -4.0...",,,,NaT,2026-06-23 10:07:40


In [27]:
rows = cim_db.get_rows('cim_pi_glass_202605', {'pi_type':'API', 'aoi':'CAAOI202'})    
bpi_df = pd.DataFrame(rows)
print(len(bpi_df))
bpi_df

9130


,sheet_id_chip_id,test_time,model_no,op_id,abbr_cat,recipe_id,cassette_id,aoi,total_defect_qty,defect_size_o_qty,defect_size_l_qty,defect_size_m_qty,defect_size_s_qty,line_id,pi_time,pi_hour,pi_type
0,6R6A5LD2H,2026-05-01 04:17:01,M270HAN03,PCS1,TFT,2227,CA0117,CAAOI202,59,0,0,8,51,CAPIC100,2026-05-01 02:55:51,2026-05-01 02:00:00,API
1,6R6A5LD2H,2026-05-01 04:19:05,M270HAN03,PCS1,TFT,0227,CA0117,CAAOI202,169,0,0,33,136,CAPIC100,2026-05-01 02:55:51,2026-05-01 02:00:00,API
2,6R6A5LD2F,2026-05-01 04:30:04,M270HAN03,PCS1,TFT,2227,CA0117,CAAOI202,54,0,1,10,43,CAPIC100,2026-05-01 02:55:51,2026-05-01 02:00:00,API
3,6R6A5LD2G,2026-05-01 04:23:14,M270HAN03,PCS1,TFT,2227,CA0117,CAAOI202,43,0,1,8,34,CAPIC100,2026-05-01 02:55:51,2026-05-01 02:00:00,API
4,6R6A5LD2G,2026-05-01 04:25:04,M270HAN03,PCS1,TFT,0227,CA0117,CAAOI202,234,0,0,16,218,CAPIC100,2026-05-01 02:55:51,2026-05-01 02:00:00,API
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9125,5H6A5101D,2026-06-01 01:56:38,B160UAN4A,PCS1,TFT,2251,CA0315,CAAOI202,8,0,0,4,4,CAPIC600,2026-05-31 20:47:43,2026-05-31 20:00:00,API
9126,5H6A5101C,2026-06-01 02:01:55,B160UAN4A,PCS1,TFT,0251,CA0315,CAAOI202,59,0,2,18,39,CAPIC600,2026-05-31 20:47:43,2026-05-31 20:00:00,API
9127,5H6A5101B,2026-06-01 02:06:36,B160UAN4A,PCS1,TFT,0251,CA0315,CAAOI202,1,0,0,1,0,CAPIC600,2026-05-31 20:47:43,2026-05-31 20:00:00,API
9128,5H6A5101A,2026-06-01 02:10:26,B160UAN4A,PCS1,TFT,0251,CA0315,CAAOI202,15,0,0,8,7,CAPIC600,2026-05-31 20:47:43,2026-05-31 20:00:00,API


In [3]:
df = inspec_db.get_table('inspection_summary_table_202606')
print(len(df))
df.drop_duplicates(subset=['SHEET_ID','SCAN_ENDTIME'], inplace=True)
print(len(df))
print(len(df.groupby(['SHEET_ID','SCAN_ENDTIME'])))
df.head()

148421
133681
133681


,CHIP_COUNT,CHIP_JUDGE,CHIP_OK_COUNT,DEFECT,FAB,MODEL_NO,RECIPE_NAME,RUN_ID,SCAN_ENDTIME,SCAN_STARTTIME,SHEET_ID,STAGE,TOOL_ID,TOTAL_DEFECT_COUNT,TYPE
0,18.0,OK,17.0,True,L6A,G215HVN01,G215HVN01-CF,-2116398710.0,2026-06-01 00:00:08,2026-05-31 23:59:46.000,AL5H151YP5,CELL,CAPIC207,5.0,CF
1,18.0,OK,18.0,True,L6A,G215HVN01,G215HVN01-CF,-2116398711.0,2026-06-01 00:01:02,2026-06-01 00:00:39.000,AL5H151YP4,CELL,CAPIC207,4.0,CF
2,36.0,NG,30.0,False,L6A,B140UAN08,B140UAN08-TFT,-2116398695.0,2026-06-01 00:00:15,2026-05-31 23:59:52.000,4C6A51T62,CELL,CAPIC507,8.0,TFT
3,36.0,OK,31.0,True,L6A,B140UAN08,B140UAN08-TFT,-2116398694.0,2026-06-01 00:01:13,2026-06-01 00:00:51.000,4C6A51T63,CELL,CAPIC507,9.0,TFT
4,18.0,OK,18.0,True,L6A,G215HVN01,G215HVN01-CF,-2116398682.0,2026-06-01 00:07:13,2026-06-01 00:06:51.000,AL5H151YML,CELL,CAPIC207,3.0,CF


In [9]:
glds = [v for v in df['SHEET_ID'].unique()  if v in cim_df['sheet_id_chip_id'].unique()]

In [ ]:
#pd.DataFrame(glds).to_csv('insepc_glds.csv')

In [5]:
cim_df = cim_db.get_table('cim_pi_glass_202606')
print(len(cim_df))
cim_df.drop_duplicates(subset=['sheet_id_chip_id', 'pi_type'], inplace=True)
print(len(cim_df))
print(len(cim_df.groupby(['sheet_id_chip_id', 'pi_type'])))
cim_df.head()

15992
11029
11029


,sheet_id_chip_id,test_time,model_no,op_id,abbr_cat,recipe_id,cassette_id,aoi,total_defect_qty,defect_size_o_qty,defect_size_l_qty,defect_size_m_qty,defect_size_s_qty,line_id,pi_time,pi_hour,pi_type
0,AL5H151Z3K,2026-05-31 19:09:01,G215HVN01_VG,PI AOI,CF,608,CA0376,CAPIT203,172,0,3,16,153,CAPIC200,2026-06-01 00:33:11,2026-06-01 00:00:00,BPI
1,AL5H151Z3G,2026-05-31 19:13:11,G215HVN01_VG,PI AOI,CF,608,CA0376,CAPIT203,167,0,1,8,158,CAPIC200,2026-06-01 00:33:11,2026-06-01 00:00:00,BPI
2,AL5H151Z3M,2026-05-31 19:17:19,G215HVN01_VG,PI AOI,CF,608,CA0376,CAPIT203,155,0,4,12,139,CAPIC200,2026-06-01 00:33:11,2026-06-01 00:00:00,BPI
3,AL5H151Z3A,2026-05-31 19:21:22,G215HVN01_VG,PI AOI,CF,608,CA0376,CAPIT203,181,1,11,23,146,CAPIC200,2026-06-01 00:33:11,2026-06-01 00:00:00,BPI
4,YH6A5U84G,2026-06-01 00:35:12,B160UAV01,PCS1,TFT,2283,CA0364,CAAOI202,23,0,0,7,16,CAPIC400,2026-06-01 00:13:18,2026-05-31 23:00:00,API


In [ ]:
db = MySQLConnet('piaoi_density')
for tbn in db.list_tables():
    if '202606' in tbn or '202607' in tbn:
        print(tbn)
        #db.drop_table(tbn)

2026-07-01 14:16:30,245 - INFO - [list_tables] 成功取得資料表名稱，共 23 張表。


density_code_summary_202606


2026-07-01 14:16:32,724 - INFO - [drop_table] 資料表 'density_code_summary_202606' 已刪除.


density_code_summary_202607


2026-07-01 14:16:33,735 - INFO - [drop_table] 資料表 'density_code_summary_202607' 已刪除.


...

2026-07-01 14:16:34,814 - INFO - [drop_table] 資料表 'density_recipe_summary_202607' 已刪除.


density_tab_summary_202606


2026-07-01 14:16:35,361 - INFO - [drop_table] 資料表 'density_tab_summary_202606' 已刪除.


density_tab_summary_202607


2026-07-01 14:16:35,847 - INFO - [drop_table] 資料表 'density_tab_summary_202607' 已刪除.


In [13]:
cim_db.get_rows('cim_pi_glass_202606', {'aoi':'CAAOI300'})

[{'sheet_id_chip_id': 'ZM6A5ZW2A',
  'test_time': datetime.datetime(2026, 6, 1, 8, 16, 28),
  'model_no': 'M270QAN06_VX',
  'op_id': 'PCS1',
  'abbr_cat': 'TFT',
  'recipe_id': '03024',
  'cassette_id': 'CA0128',
  'aoi': 'CAAOI300',
  'total_defect_qty': 640,
  'defect_size_o_qty': 0,
  'defect_size_l_qty': 0,
  'defect_size_m_qty': 0,
  'defect_size_s_qty': 640,
  'line_id': 'CAPIC700',
  'pi_time': datetime.datetime(2026, 6, 1, 5, 47, 48),
  'pi_hour': datetime.datetime(2026, 6, 1, 5, 0),
  'pi_type': 'API'},
 {'sheet_id_chip_id': 'ZG6A53S2U',
  'test_time': datetime.datetime(2026, 6, 1, 9, 26, 52),
  'model_no': 'M270QAN06_VX',
  'op_id': 'PCS1',
  'abbr_cat': 'TFT',
  'recipe_id': '03024',
  'cassette_id': 'CA0128',
  'aoi': 'CAAOI300',
  'total_defect_qty': 1772,
  'defect_size_o_qty': 0,
  'defect_size_l_qty': 0,
  'defect_size_m_qty': 0,
  'defect_size_s_qty': 1772,
  'line_id': 'CAPIC700',
  'pi_time': datetime.datetime(2026, 6, 1, 5, 47, 48),
  'pi_hour': datetime.datetime(20

In [22]:
rows = cim_db.get_rows('cim_defect_202606_aoi100_capic200', {'sheet_id_chip_id':'AL5HZ60L39'})
print(len(rows))


rows = cim_db.get_rows('cim_defect_202606_aoi100_capic200', {'sheet_id_chip_id':'AL5HZ60L39', 'adc_def_code': 'others'})
print(len(rows))

2026-07-01 16:30:18,017 - INFO - [get_rows] 'cim_defect_202606_aoi100_capic200' 無符合條件資料.


76
0


In [ ]:
row_in={'pi_hour': '26-06-30 20', 'line_id': 'CAPIC200', 'aoi': 'aoi100', 'model': 'M215HAN01', 'glass_type': 'CF', 'recipe_id': '117', 'adc_def_code': 'others', 'glass': 'AL5HZ60L38,AL5HZ60L39,AL5HZ60L3A,AL5HZ60L3B,AL5HZ60L3C,AL5HZ60L3F,AL5HZ60L3H,AL5HZ60L3K,AL5HZ60L49,AL5HZ60LD9,AL5HZ60LDR,AL5HZ60LDT,AL5HZ60LDY,AL5HZ60LDZ,AL5HZ60LGB,AL5HZ60LGD,AL5HZ60LJF,AL5HZ60LJK,AL5HZ60LJM,AL5HZ60LJQ,AL5HZ60LJV,AL5HZ60M4G,AL5HZ60MMR,AL5HZ60MZS,AL5HZ60MZT,AL5HZ60N0T,AL5HZ60N0X,AL5HZ60N0Z', 'glass_size_detail': '{"AL5HZ60L38": {"S": 56, "M": 0, "L": 0, "O": 0, "T": 56}, "AL5HZ60L39": {"S": 30, "M": 0, "L": 0, "O": 0, "T": 30}, "AL5HZ60L3A": {"S": 55, "M": 0, "L": 0, "O": 0, "T": 55}, "AL5HZ60L3B": {"S": 57, "M": 0, "L": 0, "O": 0, "T": 57}, "AL5HZ60L3C": {"S": 39, "M": 1, "L": 0, "O": 0, "T": 40}, "AL5HZ60L3F": {"S": 42, "M": 0, "L": 0, "O": 0, "T": 42}, "AL5HZ60L3H": {"S": 34, "M": 0, "L": 0, "O": 0, "T": 34}, "AL5HZ60L3K": {"S": 52, "M": 0, "L": 0, "O": 0, "T": 52}, "AL5HZ60L49": {"S": 43, "M": 0, "L": 0, "O": 0, "T": 43}, "AL5HZ60LD9": {"S": 63, "M": 0, "L": 0, "O": 0, "T": 63}, "AL5HZ60LDR": {"S": 136, "M": 86, "L": 0, "O": 0, "T": 222}, "AL5HZ60LDT": {"S": 101, "M": 30, "L": 0, "O": 0, "T": 131}, "AL5HZ60LDY": {"S": 34, "M": 0, "L": 0, "O": 0, "T": 34}, "AL5HZ60LDZ": {"S": 20, "M": 0, "L": 0, "O": 0, "T": 20}, "AL5HZ60LGB": {"S": 41, "M": 0, "L": 0, "O": 0, "T": 41}, "AL5HZ60LGD": {"S": 20, "M": 0, "L": 0, "O": 0, "T": 20}, "AL5HZ60LJF": {"S": 48, "M": 0, "L": 0, "O": 0, "T": 48}, "AL5HZ60LJK": {"S": 20, "M": 0, "L": 0, "O": 0, "T": 20}, "AL5HZ60LJM": {"S": 58, "M": 0, "L": 0, "O": 0, "T": 58}, "AL5HZ60LJQ": {"S": 59, "M": 0, "L": 0, "O": 0, "T": 59}, "AL5HZ60LJV": {"S": 62, "M": 0, "L": 0, "O": 0, "T": 62}, "AL5HZ60M4G": {"S": 69, "M": 0, "L": 0, "O": 0, "T": 69}, "AL5HZ60MMR": {"S": 52, "M": 0, "L": 0, "O": 0, "T": 52}, "AL5HZ60MZS": {"S": 44, "M": 0, "L": 0, "O": 0, "T": 44}, "AL5HZ60MZT": {"S": 62, "M": 10, "L": 0, "O": 0, "T": 72}, "AL5HZ60N0T": {"S": 66, "M": 1, "L": 0, "O": 0, "T": 67}, "AL5HZ60N0X": {"S": 21, "M": 0, "L": 0, "O": 0, "T": 21}, "AL5HZ60N0Z": {"S": 35, "M": 0, "L": 0, "O": 0, "T": 35}}'}
#INFO - [defect_map] query pi_dt=2026-06-30 20:00:00, start_dt=2026-06-27 20:00:00, end_dt=2026-07-03 20:00:00, months=['202606', '202607'], aoi=aoi100, line_id=CAPIC200, def_code=others, glds=28

In [ ]:
base_keys={'adc_def_code': 'others', 'sheet_id_chip_id':'3D6A56S9A', 'recipe_id': '117'}
in_values=['AL5HZ60N0Z', 'AL5HZ60L38', 'AL5HZ60LDZ', 'AL5HZ60L3K', 'AL5HZ60LJF', 'AL5HZ60L49', 'AL5HZ60LJK', 'AL5HZ60L3F', 'AL5HZ60LJQ', 'AL5HZ60LDT', 'AL5HZ60LD9', 'AL5HZ60L3B', 'AL5HZ60LDR', 'AL5HZ60LJV', 'AL5HZ60LDY', 'AL5HZ60MZT', 'AL5HZ60MMR', 'AL5HZ60L3A', 'AL5HZ60LJM', 'AL5HZ60MZS', 'AL5HZ60M4G', 'AL5HZ60N0T', 'AL5HZ60L3C', 'AL5HZ60L3H', 'AL5HZ60L39', 'AL5HZ60LGB', 'AL5HZ60N0X', 'AL5HZ60LGD']

In [28]:
t = 0
for gld in in_values:
    print(gld, '----')
    rows = cim_db.get_rows('cim_defect_202606_aoi100_capic200', {'sheet_id_chip_id':gld})
    gls_df = pd.DataFrame(rows)
    print('1 total:', len(rows))
    for (tt, ), gld_rows in gls_df.groupby(['test_time']):
        print('test_time', tt)
        print('2. total:', len(gld_rows))
        t += len(rows)
        for (def_code, ), defs in gld_rows.groupby((['adc_def_code'])):
            print(def_code, len(defs))

AL5HZ60N0Z ----
1 total: 65
test_time 2026-06-30 22:57:02
2. total: 65
NPI_CF 2
NPI_OIL 1
NPI_TFT 1
Not_Found 23
OP_Defect 9
PI_Gel 2
PI_Spot_NP 1
Polymer 4
SPS 1
SSIU_Polymer 6
nan 15
AL5HZ60L38 ----
1 total: 95
test_time 2026-06-30 21:58:31
2. total: 95
NPI_CF 3
NPI_OIL 1
Not_Found 21
OP_Defect 13
PIS With Particle 1
Polymer 5
SSIU_Polymer 6
nan 45
AL5HZ60LDZ ----
1 total: 67
test_time 2026-06-30 22:30:53
2. total: 67
NPI_CF 6
Not_Found 11
OP_Defect 9
PI_Gel 4
Polymer 5
SPS 3
SSIU_Polymer 12
nan 17
AL5HZ60L3K ----
1 total: 105
test_time 2026-06-30 22:14:35
2. total: 105
NPI_CF 2
Not_Found 29
OP_Defect 6
PI_Gel 1
Polymer 2
SPS 2
SSIU_Polymer 8
nan 55
AL5HZ60LJF ----
1 total: 79
test_time 2026-06-30 21:32:16
2. total: 79
NPI_CF 2
Not_Found 27
OP_Defect 6
PI_Gel 6
SSIU_Polymer 9
nan 29
AL5HZ60L49 ----
1 total: 84
test_time 2026-06-30 22:34:10
2. total: 84
NPI_CF 3
NPI_OIL 2
Not_Found 24
OP_Defect 8
PI_Gel 2
Polymer 4
SPS 1
SSIU_Polymer 6
nan 34
AL5HZ60LJK ----
1 total: 71
test_time 2026

In [26]:
t

2652

In [4]:


for (gld, ) ,bs_rows in bpi_same_cf_df.groupby(['glass_id']):
    print('search gld: ', gld, len(bs_rows))
    print(bs_rows[['bpi_pi_time', 'bpi_scan_time','bpi_defect_count',  'api_scan_time', 'api_defect_count', 'glass_side']].drop_duplicates())
    cim_rows = cim_db.get_rows(cim_tbn, {'sheet_id_chip_id':gld})
    cim_df = pd.DataFrame(cim_rows)
    print('aoi cim_db pi_glass: ', len(cim_df))
    print(cim_df[['test_time', 'model_no', 'abbr_cat', 'aoi', 'total_defect_qty', 'line_id', 'pi_time',  'pi_type']].drop_duplicates())

    inspec_gld_df = inspec_df[inspec_df['glass'].str.contains(gld)][['pi_hour',  'line_id', 'model', 'glass_type','glass']]
    print('insepction api raws:', len(inspec_gld_df))
    print(inspec_gld_df.drop_duplicates())
    
    inspec_rows = inspec_db.get_rows('inspection_summary_table_202606', {'SHEET_ID':gld})
    inspec_rows_df = pd.DataFrame(inspec_rows)
    print('insepction defect raws:', len(inspec_rows_df))
    if not inspec_rows_df.empty:
        print(inspec_rows_df)
#

search gld:  AH0EQ604SA 1
            bpi_pi_time       bpi_scan_time  bpi_defect_count  \
769 2026-06-29 18:12:33 2026-06-29 07:32:14              1523   

          api_scan_time  api_defect_count glass_side  
769 2026-06-29 21:59:09                22         CF  
aoi cim_db pi_glass:  2
            test_time      model_no abbr_cat       aoi  total_defect_qty  \
0 2026-06-29 07:32:14     M170ETN1A       CF  CAAOI202              1523   
1 2026-06-29 21:59:09  M170ETN1A_VG       CF  CAPIT203                22   

    line_id             pi_time pi_type  
0  CAPIC700 2026-06-29 18:12:33     BPI  
1  CAPIC700 2026-06-29 18:12:33     API  
insepction api raws: 0
Empty DataFrame
Columns: [pi_hour, line_id, model, glass_type, glass]
Index: []


2026-06-30 15:57:09,954 - INFO - [get_rows] 'inspection_summary_table_202606' 無符合條件資料.


insepction defect raws: 0
search gld:  AH0EQ604SC 1
            bpi_pi_time       bpi_scan_time  bpi_defect_count  \
770 2026-06-29 18:12:33 2026-06-29 06:53:29              1417   

          api_scan_time  api_defect_count glass_side  
770 2026-06-29 22:08:09                29         CF  
aoi cim_db pi_glass:  2
            test_time      model_no abbr_cat       aoi  total_defect_qty  \
0 2026-06-29 06:53:29     M170ETN1A       CF  CAAOI202              1417   
1 2026-06-29 22:08:09  M170ETN1A_VG       CF  CAPIT203                29   

    line_id             pi_time pi_type  
0  CAPIC700 2026-06-29 18:12:33     BPI  
1  CAPIC700 2026-06-29 18:12:33     API  
insepction api raws: 0
Empty DataFrame
Columns: [pi_hour, line_id, model, glass_type, glass]
Index: []


2026-06-30 15:57:34,211 - INFO - [get_rows] 'inspection_summary_table_202606' 無符合條件資料.


insepction defect raws: 0
search gld:  AH0EQ604TG 1
            bpi_pi_time       bpi_scan_time  bpi_defect_count  \
771 2026-06-29 18:12:33 2026-06-29 07:45:11              1480   

          api_scan_time  api_defect_count glass_side  
771 2026-06-29 21:56:08                27         CF  
aoi cim_db pi_glass:  2
            test_time      model_no abbr_cat       aoi  total_defect_qty  \
0 2026-06-29 07:45:11     M170ETN1A       CF  CAAOI202              1480   
1 2026-06-29 21:56:08  M170ETN1A_VG       CF  CAPIT203                27   

    line_id             pi_time pi_type  
0  CAPIC700 2026-06-29 18:12:33     BPI  
1  CAPIC700 2026-06-29 18:12:33     API  
insepction api raws: 0
Empty DataFrame
Columns: [pi_hour, line_id, model, glass_type, glass]
Index: []


...

insepction defect raws: 0
search gld:  AR0HG61GL9 1
            bpi_pi_time       bpi_scan_time  bpi_defect_count  \
191 2026-06-13 13:50:32 2026-06-13 10:53:52               223   

          api_scan_time  api_defect_count glass_side  
191 2026-06-13 17:26:40                79         CF  
aoi cim_db pi_glass:  2
            test_time      model_no abbr_cat       aoi  total_defect_qty  \
0 2026-06-13 10:53:52  M270HVN0G_VX       CF  CAPIT203               223   
1 2026-06-13 17:26:40  M270HVN0G_VX       CF  CAPIT203                79   

    line_id             pi_time pi_type  
0  CAPIC500 2026-06-13 13:50:32     BPI  
1  CAPIC500 2026-06-13 13:50:32     API  
insepction api raws: 1
                 pi_hour   line_id      model glass_type  \
1849 2026-06-13 13:00:00  CAPIC507  M270HVN0G         CF   

                                                  glass  
1849  AR0HG61HN7,AR0HG61HNA,AR0HG61HLY,AR0HG61HN8,AR...  
insepction defect raws: 1
  CHIP_COUNT CHIP_JUDGE CHIP_OK_COUNT DEFE

2026-06-30 15:58:21,291 - INFO - [get_rows] 'inspection_summary_table_202606' 無符合條件資料.


insepction api raws: 0
Empty DataFrame
Columns: [pi_hour, line_id, model, glass_type, glass]
Index: []
insepction defect raws: 0
search gld:  AR0HG61GLD 1
            bpi_pi_time       bpi_scan_time  bpi_defect_count  \
195 2026-06-13 13:50:32 2026-06-13 11:11:35               193   

          api_scan_time  api_defect_count glass_side  
195 2026-06-13 16:00:46               103         CF  
aoi cim_db pi_glass:  2
            test_time      model_no abbr_cat       aoi  total_defect_qty  \
0 2026-06-13 11:11:35  M270HVN0G_VX       CF  CAPIT203               193   
1 2026-06-13 16:00:46  M270HVN0G_VX       CF  CAPIT203               103   

    line_id             pi_time pi_type  
0  CAPIC500 2026-06-13 13:50:32     BPI  
1  CAPIC500 2026-06-13 13:50:32     API  
insepction api raws: 1
                 pi_hour   line_id      model glass_type  \
1849 2026-06-13 13:00:00  CAPIC507  M270HVN0G         CF   

                                                  glass  
1849  AR0HG61HN7,AR0HG6

2026-06-30 15:58:24,257 - INFO - [get_rows] 'inspection_summary_table_202606' 無符合條件資料.


insepction defect raws: 0
search gld:  AR0HG61GT5 1
            bpi_pi_time       bpi_scan_time  bpi_defect_count  \
207 2026-06-13 13:50:32 2026-06-13 09:33:37               187   

          api_scan_time  api_defect_count glass_side  
207 2026-06-13 18:34:52                82         CF  
aoi cim_db pi_glass:  2
            test_time      model_no abbr_cat       aoi  total_defect_qty  \
0 2026-06-13 09:33:37  M270HVN0G_VX       CF  CAPIT203               187   
1 2026-06-13 18:34:52  M270HVN0G_VX       CF  CAPIT203                82   

    line_id             pi_time pi_type  
0  CAPIC500 2026-06-13 13:50:32     BPI  
1  CAPIC500 2026-06-13 13:50:32     API  
insepction api raws: 1
                 pi_hour   line_id      model glass_type  \
1840 2026-06-13 12:00:00  CAPIC507  M270HVN0G         CF   

                                                  glass  
1840  AR0HG61JHL,AR0HG61JHK,AR0HG61JHJ,AR0HG61JHF,AR...  
insepction defect raws: 1
  CHIP_COUNT CHIP_JUDGE CHIP_OK_COUNT DEFE

In [27]:
gld = 'YH6A56B4D'
db = MySQLConnet('piaoi_inspection_density')
tbn = 'inspection_summary_table_202606'
#tbn = 'inspection_raw_table_202606'
#tbn = 'incoming_glass_summary_202606'
#rows = db.get_rows(tbn, {'TOOL_ID':'CAPIC407'})
#rows
df = db.get_table('inspection_api_summary_202606')
df[df['glass'].str.contains(gld)]

,pi_hour,shift_day,shift_week,shift_month,shift_start,shift_end,line_id,model,glass_type,maingroup_glass_count,...,small_defect_count,middle_defect_count,large_defect_count,over_defect_count,glass,glass_size_detail,comment,action,Editor,modify_time
3322,2026-06-23 04:00:00,2026-06-22,2026W26,202606,2026-06-23 04:30:00,2026-06-23 05:30:00,CAPIC407,B160UAV01,TFT,28,...,9,18,26,6,"YH6A56B4A,YH6A56B4B,YH6A56B4C,YH6A56B4D,YH6A56...","[{""glass_id"": ""YH6A56B4A"", ""S"": 0, ""M"": 1, ""L""...",,,,2026-06-23 08:27:26


In [28]:
df.columns

Index(['pi_hour', 'shift_day', 'shift_week', 'shift_month', 'shift_start',
       'shift_end', 'line_id', 'model', 'glass_type', 'maingroup_glass_count',
       'maingroup_defect_count', 'maingroup_density',
       'defect_code_glass_count', 'small_defect_count', 'middle_defect_count',
       'large_defect_count', 'over_defect_count', 'glass', 'glass_size_detail',
       'comment', 'action', 'Editor', 'modify_time'],
      dtype='object')

In [30]:

tbn = 'inspection_summary_table_202606'
tbn = 'inspection_raw_table_202606'
rows = db.get_rows(tbn, {'SHEET_ID':gld})
df = pd.DataFrame(rows)
print(len(rows))
df.iloc[0,:].to_dict()

3


{'COORD_X': '-846.8',
 'COORD_Y': '-174.7967',
 'DEFECT': 'False',
 'DEFECT_AREA': '',
 'DEFECT_ID': '1.0',
 'DEFECT_SIZE_TYPE': 'M',
 'FAB': 'L6A',
 'FRONT_REVERSE': '1P',
 'IMG_URL': '',
 'RECIPE_NAME': 'B160UAV01-TFT',
 'RUN_ID': '-2116295107.0',
 'SCAN_ENDTIME': Timestamp('2026-06-23 04:46:42'),
 'SCAN_STARTTIME': '2026-06-23T04:46:19',
 'SHEET_ID': 'YH6A56B4D',
 'SP': 'N',
 'STAGE': 'CELL',
 'TOOL_ID': 'CAPIC407',
 'TOTAL_DEFECT_COUNT': '3.0'}

In [25]:
#db = MySQLConnet('piaoi_density')
aoi_rows = cim_db.get_rows('cim_pi_glass_202606', {'sheet_id_chip_id':gld})
aoi_df=  pd.DataFrame(aoi_rows)
print(len(aoi_df))
aoi_df

3


,sheet_id_chip_id,test_time,model_no,op_id,abbr_cat,recipe_id,cassette_id,aoi,total_defect_qty,defect_size_o_qty,defect_size_l_qty,defect_size_m_qty,defect_size_s_qty,line_id,pi_time,pi_hour,pi_type
0,YH6A56B4D,2026-06-22 23:30:39,B160UAV01,PCS1,TFT,4283,AA0268,CAAOI202,157,0,0,74,83,CAPIC400,2026-06-23 05:59:05,2026-06-23 05:00:00,BPI
1,YH6A56B4D,2026-06-23 07:45:21,B160UAV01,PCS1,TFT,0283,CA0315,CAAOI202,312,0,0,110,202,CAPIC400,2026-06-23 05:59:05,2026-06-23 05:00:00,API
2,YH6A56B4D,2026-06-23 07:43:33,B160UAV01,PCS1,TFT,2283,CA0315,CAAOI202,20,0,0,11,9,CAPIC400,2026-06-23 05:59:05,2026-06-23 05:00:00,API


In [26]:
aoi_df.columns

Index(['sheet_id_chip_id', 'test_time', 'model_no', 'op_id', 'abbr_cat',
       'recipe_id', 'cassette_id', 'aoi', 'total_defect_qty',
       'defect_size_o_qty', 'defect_size_l_qty', 'defect_size_m_qty',
       'defect_size_s_qty', 'line_id', 'pi_time', 'pi_hour', 'pi_type'],
      dtype='object')

In [6]:
db = MySQLConnet('cim_cell_aoi_to_array')
tbn = 'inspection_summary_table_202606'
tbn = 'inspection_raw_table_202606'
#tbn = 'incoming_glass_summary_202606'
df = db.get_table(tbn)
#print(df['judge'].unique())
df.head()

ValueError: Table inspection_raw_table_202606 not found

In [2]:
db = MySQLConnet('piaoi_inspection_density')
tbn = 'inspection_summary_table_202606'
tbn = 'inspection_raw_table_202606'
df = db.get_table(tbn)
df.head()

,COORD_X,COORD_Y,DEFECT,DEFECT_AREA,DEFECT_ID,DEFECT_SIZE_TYPE,FAB,FRONT_REVERSE,IMG_URL,RECIPE_NAME,RUN_ID,SCAN_ENDTIME,SCAN_STARTTIME,SHEET_ID,SP,STAGE,TOOL_ID,TOTAL_DEFECT_COUNT
0,-55.645,-1327.566,True,,1.0,S,L6A,1P,http://10.97.138.179/20260601/AL5H151YP4/1.jpg,G215HVN01-CF,-2116398711.0,2026-06-01 00:01:02,2026-06-01T00:00:39,AL5H151YP4,N,CELL,CAPIC207,4.0
1,-1391.611,-352.24,True,,2.0,S,L6A,1N,http://10.97.138.179/20260601/AL5H151YP4/2.jpg,G215HVN01-CF,-2116398711.0,2026-06-01 00:01:02,2026-06-01T00:00:39,AL5H151YP4,N,CELL,CAPIC207,4.0
2,-310.482,-1053.6,True,,3.0,S,L6A,1P,http://10.97.138.179/20260601/AL5H151YP4/3.jpg,G215HVN01-CF,-2116398711.0,2026-06-01 00:01:02,2026-06-01T00:00:39,AL5H151YP4,N,CELL,CAPIC207,4.0
3,-422.042,-1298.244,True,,4.0,S,L6A,1N,http://10.97.138.179/20260601/AL5H151YP4/4.jpg,G215HVN01-CF,-2116398711.0,2026-06-01 00:01:02,2026-06-01T00:00:39,AL5H151YP4,N,CELL,CAPIC207,4.0
4,-1188.963,-892.938,True,,1.0,S,L6A,1P,http://10.97.138.179/20260601/AL5H151YP5/1.jpg,G215HVN01-CF,-2116398710.0,2026-06-01 00:00:08,2026-05-31T23:59:46,AL5H151YP5,N,CELL,CAPIC207,5.0


In [ ]:
['COORD_X', 'COORD_Y','DEFECT_SIZE_TYPE','IMG_URL', 'RECIPE_NAME',
       'RUN_ID', 'SCAN_ENDTIME','SHEET_ID', 'SP', 'STAGE',
       'TOOL_ID', 'TOTAL_DEFECT_COUNT']

Index(['COORD_X', 'COORD_Y', 'DEFECT', 'DEFECT_AREA', 'DEFECT_ID',
       'DEFECT_SIZE_TYPE', 'FAB', 'FRONT_REVERSE', 'IMG_URL', 'RECIPE_NAME',
       'RUN_ID', 'SCAN_ENDTIME', 'SCAN_STARTTIME', 'SHEET_ID', 'SP', 'STAGE',
       'TOOL_ID', 'TOTAL_DEFECT_COUNT'],
      dtype='object')

In [3]:
df.iloc[0,:].to_dict()

{'COORD_X': '-55.645',
 'COORD_Y': '-1327.566',
 'DEFECT': 'True',
 'DEFECT_AREA': '',
 'DEFECT_ID': '1.0',
 'DEFECT_SIZE_TYPE': 'S',
 'FAB': 'L6A',
 'FRONT_REVERSE': '1P',
 'IMG_URL': 'http://10.97.138.179/20260601/AL5H151YP4/1.jpg',
 'RECIPE_NAME': 'G215HVN01-CF',
 'RUN_ID': '-2116398711.0',
 'SCAN_ENDTIME': Timestamp('2026-06-01 00:01:02'),
 'SCAN_STARTTIME': '2026-06-01T00:00:39',
 'SHEET_ID': 'AL5H151YP4',
 'SP': 'N',
 'STAGE': 'CELL',
 'TOOL_ID': 'CAPIC207',
 'TOTAL_DEFECT_COUNT': '4.0'}

In [ ]:
{'CHIP_COUNT': '18.0',
 
 'MODEL_NO': 'G215HVN01',
 'RECIPE_NAME': 'G215HVN01-CF',
 'RUN_ID': '-2116398710.0',
 'SCAN_ENDTIME': Timestamp('2026-06-01 00:00:08'),
 'SCAN_STARTTIME': '2026-05-31 23:59:46.000',
 'SHEET_ID': 'AL5H151YP5',
 'STAGE': 'CELL',
 'TOOL_ID': 'CAPIC207',
 'TOTAL_DEFECT_COUNT': '5.0',
 'TYPE': 'CF'}

In [6]:
for (gld, glass_type, ),rows in  df.groupby(['SHEET_ID', 'TYPE']):
    if len(rows['TOOL_ID'].unique())>1:
        print(gld, glass_type, rows['TOOL_ID'].unique())

20260604221728 TFT ['CAPIC507' 'CAPIC407']
4C6A62G21 TFT ['CAPIC507' 'CAPIC107']
4C6A62G22 TFT ['CAPIC507' 'CAPIC107']
4C6A62G23 TFT ['CAPIC507' 'CAPIC107']
4C6A62G24 TFT ['CAPIC507' 'CAPIC107']
4C6A62G2A TFT ['CAPIC507' 'CAPIC107']
4C6A62G2B TFT ['CAPIC507' 'CAPIC107']
4C6A62G2D TFT ['CAPIC507' 'CAPIC107']
4C6A62G2E TFT ['CAPIC507' 'CAPIC107']
4C6A62G2G TFT ['CAPIC507' 'CAPIC107']
4C6A62G2J TFT ['CAPIC507' 'CAPIC107']
4C6A62G2K TFT ['CAPIC507' 'CAPIC107']
4C6A62G2M TFT ['CAPIC507' 'CAPIC107']
4C6A62G2N TFT ['CAPIC507' 'CAPIC107']
4C6A62G2P TFT ['CAPIC507' 'CAPIC107']
4C6A62G2T TFT ['CAPIC507' 'CAPIC107']
4C6A62G2U TFT ['CAPIC507' 'CAPIC107']
4C6A62G2V TFT ['CAPIC507' 'CAPIC107']
4C6A62G2W TFT ['CAPIC507' 'CAPIC107']
4C6A62G2X TFT ['CAPIC507' 'CAPIC107']
4C6A62H71 TFT ['CAPIC507' 'CAPIC107']
4C6A62H72 TFT ['CAPIC507' 'CAPIC107']
4C6A62H73 TFT ['CAPIC507' 'CAPIC107']
4C6A62H74 TFT ['CAPIC507' 'CAPIC107']
4C6A62H7A TFT ['CAPIC507' 'CAPIC107']
4C6A62H7B TFT ['CAPIC507' 'CAPIC107']
4C6A62H

In [ ]:
df = cim_db.get_runs_delta_days('cim_defect_202606_aoi200_capic600', 2, 'test_time')
df.head()

,sheet_id_chip_id,chip_id,test_time,defect_size,pox_x1,pox_y1,image_file_path,image_file_name,img_file_url_path,retype_def_code,adc_def_code,pi_time,pi_hour
0,5T6A60Y0D,A,2026-06-18 06:20:37,M,379368,576681,Image/CA036602/5T6A60Y0D/,RV1_379368_576681_0.jpg,PIT/2606/18/CAAOI202/5T6A60Y0D/0620/,nan,PI_Spot_NP,2026-06-18 03:45:20,2026-06-18 03:00:00
1,5T6A60Y0D,A,2026-06-18 06:20:37,S,1111430,1444051,Image/CA036602/5T6A60Y0D/,RV1_1111430_1444051_1.jpg,PIT/2606/18/CAAOI202/5T6A60Y0D/0620/,nan,PI_Spot_NP,2026-06-18 03:45:20,2026-06-18 03:00:00
2,5T6A60Y0D,A,2026-06-18 06:20:37,S,1466114,994383,Image/CA036602/5T6A60Y0D/,RV1_1466114_994383_2.jpg,PIT/2606/18/CAAOI202/5T6A60Y0D/0620/,nan,PI_Spot_NP,2026-06-18 03:45:20,2026-06-18 03:00:00
3,5T6A60Y0D,A,2026-06-18 06:20:37,S,386825,649628,Image/CA036602/5T6A60Y0D/,RV1_386825_649628_3.jpg,PIT/2606/18/CAAOI202/5T6A60Y0D/0620/,nan,PI_Spot_NP,2026-06-18 03:45:20,2026-06-18 03:00:00
4,5T6A60Y0D,A,2026-06-18 06:20:37,S,319425,560372,Image/CA036602/5T6A60Y0D/,RV1_319425_560372_4.jpg,PIT/2606/18/CAAOI202/5T6A60Y0D/0620/,nan,PI_Spot_NP,2026-06-18 03:45:20,2026-06-18 03:00:00


In [4]:
#df = cim_db.get_runs_delta_days('cim_defect_202606_aoi200_capic600', 7, 'test_time')
#df['defect_size'].unique()
db = MySQLConnet('piaoi_bpi_same_point')
#df = db.get_table('bpi_api_summary_202604')
#df.iloc[0,:].to_dict()

In [ ]:
datetime.datetime(2026, 6, 20, 15, 0)

In [10]:
row_in={'pi_hour': '26-06-20 15', 'line_id': 'CAPIC700', 'aoi': 'aoi200', 'model': 'M270QAN06', 'glass_type': 'TFT', 'recipe_id': '2234', 'adc_def_code': 'SSIU_Polymer', 'glass': 'ZG6A64H1C,ZG6A64H1D,ZG6A64H1E,ZG6A64H1F,ZG6A64H1G', 'glass_size_detail': '{"ZG6A64H1C": {"S": 6, "M": 0, "L": 0, "O": 0, "T": 6}, "ZG6A64H1D": {"S": 15, "M": 0, "L": 0, "O": 0, "T": 15}, "ZG6A64H1E": {"S": 9, "M": 0, "L": 0, "O": 0, "T": 9}, "ZG6A64H1F": {"S": 13, "M": 0, "L": 0, "O": 0, "T": 13}, "ZG6A64H1G": {"S": 2, "M": 21, "L": 14, "O": 10, "T": 47}}'}
base_keys={'adc_def_code': 'SSIU_Polymer', 'pi_hour': "2026-06-20 15:00:00"}
tbn = 'cim_defect_202606_aoi200_capic700'
glds = {'ZG6A64H1D', 'ZG6A64H1E', 'ZG6A64H1C', 'ZG6A64H1G', 'ZG6A64H1F'}

In [13]:
cols = [ 'sheet_id_chip_id', 'adc_def_code','defect_size', 'pi_time', 'pox_x1',
       'pox_y1', 'test_time']

In [4]:
[v for v in cim_db.list_tables() if 'cim_defect' in v and '202606' in v]

2026-06-23 14:27:20,380 - INFO - [list_tables] 成功取得資料表名稱，共 179 張表。


['__stg_cim_defect_202606_aoi300_capic700_1782172498',
 'cim_defect_202606_aoi100_capic100',
 'cim_defect_202606_aoi100_capic200',
 'cim_defect_202606_aoi100_capic300',
 'cim_defect_202606_aoi100_capic400',
 'cim_defect_202606_aoi100_capic500',
 'cim_defect_202606_aoi100_capic600',
 'cim_defect_202606_aoi100_capic700',
 'cim_defect_202606_aoi100_pi000',
 'cim_defect_202606_aoi200_capic100',
 'cim_defect_202606_aoi200_capic200',
 'cim_defect_202606_aoi200_capic300',
 'cim_defect_202606_aoi200_capic400',
 'cim_defect_202606_aoi200_capic500',
 'cim_defect_202606_aoi200_capic600',
 'cim_defect_202606_aoi200_capic700',
 'cim_defect_202606_aoi200_pi000',
 'cim_defect_202606_aoi300_capic100',
 'cim_defect_202606_aoi300_capic200',
 'cim_defect_202606_aoi300_capic300',
 'cim_defect_202606_aoi300_capic400',
 'cim_defect_202606_aoi300_capic500',
 'cim_defect_202606_aoi300_capic600',
 'cim_defect_202606_aoi300_capic700',
 'cim_defect_202606_aoi300_pi000']

In [3]:
db = MySQLConnet('piaoi_density')
df = db.get_table('default_spec_table')
df[df['model'].str.contains('test')]

,line_id,model,glass_type,adc_def_code,defect_size,OOC,OOS,Editor,modify_time,drop,MODEL_TYPE,PROCESS_TYPE
7422,CAPIC100,test_0317_2,TFT,PI_Spot_NP,S,0,5,預設,2026-03-17 15:58:57,T,高階,BIG
7434,CAPIC100,test_0429,TFT,PI_Spot_NP,S,1,2,預設,2026-04-29 10:39:59,T,Normal,1
7603,CAPIC700,test_0623,TFT,NPI_CF,OLM,5,10,預設,2026-06-23 13:52:53,F,Normal,BIG


In [ ]:

tbn = 'density_code_summary_202606'
base_keys={'adc_def_code': 'SSIU_Polymer', 'pi_hour': "2026-06-20 15:00:00", 'aoi':'aoi200', 'line_id':'CAPIC700', 'model':'M270QAN06'}
"""
df_all = cim_db.get_rows_df_in(
    table_name=tbn,
    base_keys=base_keys,
    in_key="sheet_id_chip_id",
    in_values=list(glds),
)
"""
df = db.get_rows(tbn,base_keys )
df

[{'line_id': 'CAPIC700',
  'aoi': 'aoi200',
  'model': 'M270QAN06',
  'glass_type': 'TFT',
  'pi_hour': datetime.datetime(2026, 6, 20, 15, 0),
  'recipe_id': '0234',
  'adc_def_code': 'SSIU_Polymer',
  'recipe_total_glass_cnt': 5,
  'recipe_total_defect_cnt': 1651,
  'recipe_total_density': 330.2,
  'defect_cnt': 5,
  'def_glass_cnt': 1,
  'glass_cnt': 5,
  'recipe_code_density': 1.0,
  'density': 1.0,
  'small_defect_count': 3,
  'middle_defect_count': 2,
  'large_defect_count': 0,
  'over_defect_count': 0,
  'glass': 'ZG6A64H1C,ZG6A64H1D,ZG6A64H1E,ZG6A64H1F,ZG6A64H1G',
  'glass_size_detail': '{"ZG6A64H1C": {"S": 0, "M": 0, "L": 0, "O": 0, "T": 0}, "ZG6A64H1D": {"S": 0, "M": 0, "L": 0, "O": 0, "T": 0}, "ZG6A64H1E": {"S": 0, "M": 0, "L": 0, "O": 0, "T": 0}, "ZG6A64H1F": {"S": 0, "M": 0, "L": 0, "O": 0, "T": 0}, "ZG6A64H1G": {"S": 3, "M": 2, "L": 0, "O": 0, "T": 5}}',
  'comment': '',
  'action': '',
  'Editor': '',
  'modify_time': '2026-06-23 10:10:32'},
 {'line_id': 'CAPIC700',
  'ao

In [18]:
tbn = 'cim_defect_202606_aoi200_capic700'
df_all = cim_db.get_rows_df_in(
    table_name=tbn,
    base_keys=base_keys,
    in_key="sheet_id_chip_id",
    in_values=list(glds),
)
for (gld, tt), rows in df_all[cols].groupby(['sheet_id_chip_id', "test_time"]):
    print(gld, tt, len(rows))
    if gld !='ZG6A64H1G':continue
    print(rows)

2026-06-23 11:12:30,816 - INFO - [get_rows_df_in] 成功取得 'cim_defect_202606_aoi200_capic700' 共 108 筆符合條件資料.


ZG6A64H1C 2026-06-20 17:58:02 6
ZG6A64H1D 2026-06-20 17:48:48 15
ZG6A64H1E 2026-06-20 17:24:24 9
ZG6A64H1F 2026-06-20 17:13:55 13
ZG6A64H1G 2026-06-20 16:47:27 13
   sheet_id_chip_id  adc_def_code defect_size             pi_time   pox_x1  \
0         ZG6A64H1G  SSIU_Polymer           S 2026-06-20 15:43:21  1796313   
1         ZG6A64H1G  SSIU_Polymer           S 2026-06-20 15:43:21  1412045   
2         ZG6A64H1G  SSIU_Polymer           S 2026-06-20 15:43:21  1574059   
3         ZG6A64H1G  SSIU_Polymer           S 2026-06-20 15:43:21  1803645   
4         ZG6A64H1G  SSIU_Polymer           S 2026-06-20 15:43:21  1818081   
5         ZG6A64H1G  SSIU_Polymer           S 2026-06-20 15:43:21  1309441   
6         ZG6A64H1G  SSIU_Polymer           S 2026-06-20 15:43:21  1452014   
7         ZG6A64H1G  SSIU_Polymer           S 2026-06-20 15:43:21  1614355   
8         ZG6A64H1G  SSIU_Polymer           S 2026-06-20 15:43:21  1584014   
9         ZG6A64H1G  SSIU_Polymer           S 2026-06-20 

In [ ]:
glds = []
tb_data = {}
for tbn in db.list_tables():
    print(tbn)
    
    if '_202606' not in tbn:continue
    #tb_data[tbn] = db.get_table(tbn)
    #db.drop_table(tbn)
    """
    df = db.get_table(tbn)
    tb_data[tbn] = df.copy()
    for v in df[df['aoi'] =='aoi200']['glass_list']:
        glds = glds + v.split(",")
    """

2026-06-23 08:49:27,518 - INFO - [list_tables] 成功取得資料表名稱，共 7 張表。


bpi_same_point_202605
bpi_same_point_202606


2026-06-23 08:49:30,533 - INFO - [drop_table] 資料表 'bpi_same_point_202606' 已刪除.


bpi_same_point_match_detail_202605
bpi_same_point_match_detail_202606


2026-06-23 08:49:40,883 - INFO - [drop_table] 資料表 'bpi_same_point_match_detail_202606' 已刪除.


bpi_same_point_offset_summary_202605
bpi_same_point_offset_summary_202606


2026-06-23 08:49:42,079 - INFO - [drop_table] 資料表 'bpi_same_point_offset_summary_202606' 已刪除.


default_spec_table


In [5]:
glds = [v for v in set(glds)]
len(glds)

1061

In [15]:
cim_db.get_rows_df_in('cim_pi_glass_202605', {}, 'sheet_id_chip_id', ['LF6A5NW3H']).to_dict(orient='index')

2026-06-18 15:38:05,096 - INFO - [get_rows_df_in] 成功取得 'cim_pi_glass_202605' 共 1 筆符合條件資料.


{0: {'abbr_cat': 'TFT',
  'aoi': 'CAAOI202',
  'cassette_id': 'AA0500',
  'defect_size_l_qty': 85,
  'defect_size_m_qty': 95,
  'defect_size_o_qty': 0,
  'defect_size_s_qty': 19,
  'line_id': 'CAPIC100',
  'model_no': 'M215HAN01',
  'op_id': 'PCS1',
  'pi_hour': Timestamp('2026-05-04 03:00:00'),
  'pi_time': Timestamp('2026-05-04 03:31:28'),
  'pi_type': 'BPI',
  'recipe_id': '4231',
  'sheet_id_chip_id': 'LF6A5NW3H',
  'test_time': Timestamp('2026-05-03 17:24:08'),
  'total_defect_qty': 468}}

In [17]:
85+95+19+269

468

In [11]:
tbns = ['cim_defect_202605_aoi200_capic100',
            'cim_defect_202605_aoi200_capic200',
            'cim_defect_202605_aoi200_capic300',
            'cim_defect_202605_aoi200_capic400',
            'cim_defect_202605_aoi200_capic500',
            'cim_defect_202605_aoi200_capic600',
            'cim_defect_202605_aoi200_capic700',
            'cim_defect_202605_aoi200_pi000']
for tbn in tbns:
    df = cim_db.get_rows_df_in(tbn, base_keys={},in_key = 'sheet_id_chip_id',in_values=glds,empty_cols=['defect_size'])
    if df.empty:continue
    print(tbn)
    for (gld, tt, ), rows in df.groupby(['sheet_id_chip_id', 'pi_time']):
        print(gld, tt, len(rows))

2026-06-18 15:01:57,068 - INFO - [get_rows_df_in] 成功取得 'cim_defect_202605_aoi200_capic100' 共 2448 筆符合條件資料.


cim_defect_202605_aoi200_capic100
8A6A5VW1A 2026-05-30 18:13:52 1
8A6A5VW1B 2026-05-30 18:13:52 3
8A6A5VW1C 2026-05-30 18:13:52 3
LF6A5NW3G 2026-05-04 03:31:28 1
LF6A5NW3H 2026-05-04 03:31:28 269
LF6A5NW3L 2026-05-04 03:31:28 1
Y26A5RV7A 2026-05-14 13:02:28 3
Y26A5RV7B 2026-05-14 13:02:28 15
Y26A5RV7C 2026-05-14 13:02:28 1
Y26A5RV7F 2026-05-14 22:21:05 3
Y26A5RV7G 2026-05-14 22:21:05 2093
Y26A5RV7H 2026-05-14 22:21:05 1
Y26A5RV7X 2026-05-14 23:16:21 1
YL6A5059E 2026-05-27 16:01:50 17
YL6A5059F 2026-05-27 16:01:50 17
YL6A5059G 2026-05-27 16:01:50 8
YL6A5059H 2026-05-27 16:01:50 10
YL6A5ZD2E 2026-05-26 20:45:13 1


2026-06-18 15:01:57,696 - INFO - [get_rows_df_in] 'cim_defect_202605_aoi200_capic200' 無符合條件資料.
2026-06-18 15:01:58,892 - INFO - [get_rows_df_in] 成功取得 'cim_defect_202605_aoi200_capic300' 共 5 筆符合條件資料.


cim_defect_202605_aoi200_capic300
HJ6A5KT9A 2026-05-13 22:53:59 1
HJ6A5KT9J 2026-05-13 22:53:59 1
YD6A5PY0B 2026-05-04 00:39:49 1
YD6A5PY0E 2026-05-04 00:39:49 1
YD6A5PY0F 2026-05-04 00:39:49 1


2026-06-18 15:02:00,408 - INFO - [get_rows_df_in] 成功取得 'cim_defect_202605_aoi200_capic400' 共 9 筆符合條件資料.


cim_defect_202605_aoi200_capic400
VP6G3RX94 2026-05-09 18:41:26 1
VP6G4E16J 2026-05-02 07:41:38 1
VP6G4F40G 2026-05-10 01:04:54 1
VP6G4FH24 2026-05-10 01:04:54 2
VP6G4FH5W 2026-05-08 16:49:22 1
VP6G4JA84 2026-05-15 23:32:56 1
VP6G4PR7T 2026-06-04 20:50:43 1
YH6A5U88C 2026-05-31 13:35:37 1


2026-06-18 15:02:01,101 - INFO - [get_rows_df_in] 'cim_defect_202605_aoi200_capic500' 無符合條件資料.
2026-06-18 15:02:01,565 - INFO - [get_rows_df_in] 成功取得 'cim_defect_202605_aoi200_capic600' 共 1 筆符合條件資料.


cim_defect_202605_aoi200_capic600
5H6A5YR5E 2026-05-30 21:25:20 1


2026-06-18 15:02:03,517 - INFO - [get_rows_df_in] 'cim_defect_202605_aoi200_capic700' 無符合條件資料.
2026-06-18 15:02:04,174 - INFO - [get_rows_df_in] 成功取得 'cim_defect_202605_aoi200_pi000' 共 10 筆符合條件資料.


cim_defect_202605_aoi200_pi000


8A6A5VW1A 2026-05-30 18:55:15 1
8A6A5VW1B 2026-05-30 18:50:21 3
8A6A5VW1C 2026-05-30 18:46:01 3
LF6A5NW3G 2026-05-03 17:19:44 1
LF6A5NW3H 2026-05-03 17:24:08 269
LF6A5NW3L 2026-05-03 17:37:33 1
Y26A5RV7A 2026-05-12 12:08:06 1
Y26A5RV7A 2026-05-14 19:08:51 1
Y26A5RV7A 2026-05-14 19:13:06 1
Y26A5RV7B 2026-05-14 18:48:07 14
Y26A5RV7B 2026-05-14 19:00:05 1
Y26A5RV7C 2026-05-14 19:06:28 1
Y26A5RV7F 2026-05-12 12:23:43 1
Y26A5RV7F 2026-05-14 23:55:02 1
Y26A5RV7F 2026-05-14 23:58:07 1
Y26A5RV7G 2026-05-12 12:27:14 966
Y26A5RV7G 2026-05-14 23:43:14 1127
Y26A5RV7H 2026-05-14 23:37:06 1
Y26A5RV7X 2026-05-15 01:15:42 1
YL6A5059E 2026-05-27 10:44:26 16
YL6A5059E 2026-05-27 18:10:51 1
YL6A5059F 2026-05-27 10:47:34 16
YL6A5059F 2026-05-27 18:06:42 1
YL6A5059G 2026-05-27 10:50:30 8
YL6A5059H 2026-05-27 10:53:16 10
YL6A5ZD2E 2026-05-26 23:58:46 1


In [18]:
[v for v in cim_db.list_tables() if '202605_aoi200' in v]

2026-06-18 14:44:43,876 - INFO - [list_tables] 成功取得資料表名稱，共 178 張表。


['cim_defect_202605_aoi200_capic100',
 'cim_defect_202605_aoi200_capic200',
 'cim_defect_202605_aoi200_capic300',
 'cim_defect_202605_aoi200_capic400',
 'cim_defect_202605_aoi200_capic500',
 'cim_defect_202605_aoi200_capic600',
 'cim_defect_202605_aoi200_capic700',
 'cim_defect_202605_aoi200_pi000']

In [24]:
db = MySQLConnet('piaoi_inspection_density')
#rows = db.get_rows('inspection_raw_table_202606', {'TOOL_ID':'CAPIC207'})
rows = db.get_runs_delta_days('inspection_raw_table_202606', 9,'SCAN_ENDTIME')
df = rows[rows['SHEET_ID'] .isin(['AL5H151Z1P','L66A5181F','AL5HC500PC','3N6A5JD9A'])]


In [26]:
df[['SHEET_ID','COORD_X', 'COORD_Y',  'DEFECT_SIZE_TYPE', 'RECIPE_NAME', 'SCAN_ENDTIME', 'TOTAL_DEFECT_COUNT']].to_csv('inspection_data2.csv')

for (scan_time, gld, ), rows in df.groupby(['SCAN_ENDTIME', 'SHEET_ID']):
    print(scan_time, gld)
    print(rows.loc[rows.index[:3],rows.columns])
    

2026-06-01 07:46:56 AL5H151Z1P
         COORD_X    COORD_Y DEFECT DEFECT_AREA DEFECT_ID DEFECT_SIZE_TYPE  \
176080  -573.089  -1535.837   True                   1.0                S   
176081   -197.59    -888.72   True                   2.0                O   
176082  -462.718  -1083.322   True                   3.0                S   

        FAB FRONT_REVERSE                                         IMG_URL  \
176080  L6A            1N  http://10.97.138.179/20260601/AL5H151Z1P/1.jpg   
176081  L6A            1N  http://10.97.138.179/20260601/AL5H151Z1P/2.jpg   
176082  L6A            1N  http://10.97.138.179/20260601/AL5H151Z1P/3.jpg   

         RECIPE_NAME         RUN_ID        SCAN_ENDTIME       SCAN_STARTTIME  \
176080  G215HVN01-CF  -2116396399.0 2026-06-01 07:46:56  2026-06-01T07:46:33   
176081  G215HVN01-CF  -2116396399.0 2026-06-01 07:46:56  2026-06-01T07:46:33   
176082  G215HVN01-CF  -2116396399.0 2026-06-01 07:46:56  2026-06-01T07:46:33   

          SHEET_ID SP STAGE   

In [ ]:
{"OC": "testing_date",  #"scan_time"
 "PS": "testing_date",  #"scan_time"
 "PX1=MOR": "test_time", #"scan_time"
 "TAR":"ddate_ttime",    #"repair time",
 "TOS": "ddate_ttime"    #"repair time",
 }

In [ ]:
"same_point_offset"
"same_point_defect_cnt "
"same_point_rate "

In [ ]:
{
    "glass_id": "sheet_id",
    "chip_id":"chip_id",
    "model_no": "model_no",
    "testing_date":"scan_time",
    "eqp_id":"eqp_id",
    "op":"op",
    "defect_no":"defect_no",
    "defect_code":"defect_code",
    "defect_size_type":"defect_size",
    "coord_x": "ori_x",
    "coord_y": "ori_y",
    "repair_op":"repair_op",
    }

{
    "tft_lot_id": "lot_id",
    "test_time":"scan_time",
    "tft_sheet_id":"sheet_id",
    "tft_chip_id":"chip_id",
    "signal_no":"signal",
    "gate_no":"gate",
    "pox_x": "ori_x",
    "pox_y": "ori_y",
    "defect_size":"defect_size",
    "image_file_name":"image_name",
    "op_key":"op_id",
    "recipe_id":"recipe_id",
    "eqp_id":"eqp_id",
    "adc_repair_answers":"defect_code",
}


{
    "lot_id": "lot_id",
    "board_id":"sheet_id",
    "chip_id":"chip_id",
    "data_ax":"signal",
    "gate_ax":"gate",
    "dft_mode":"signal_gate_defect_code",
    "route":"route",
    "ddate_ttime":"repair time",
    "retype":"defect_code",
    "tar_judge":"defect_size",
    "x_cord": "ori_x",
    "y_cord": "ori_y",
    "tester_tool": "tester_tool",
    "image_name": "image_name"

}

{'sheet_id_chip_id': 'sheet_id',
 'test_time': "scan_time",
 'model_no': 'G215HVN01_VG',
 'abbr_cat': 'CF',
 'recipe_id': 'recipe_id',
 'cassette_id': 'cassette_id',
 'aoi': 'cell_aoi',
 'total_defect_qty': 'cell_total_defect_cnt',
 'line_id': 'cell_line_id',
 'pi_time': 'pi_time',
 'pi_type': 'cell_op'
}


"""

"""

In [4]:
{
'sheet_id_chip_id': 'sheet_id', 
'line_id': 'line_id', 
'test_time': 'test_time',  
'model_no': 'model_no',  
'abbr_cat': 'abbr_cat', 
'recipe_id': 'recipe_id', 
'cassette_id': 'cassette_id', 
'aoi': 'aoi',  
'total_defect_qty': 'total_defect_qty', 
'pi_time': 'pi_time',  
'pi_type': 'pi_type', 

}.keys()

dict_keys(['sheet_id_chip_id', 'line_id', 'test_time', 'model_no', 'abbr_cat', 'recipe_id', 'cassette_id', 'aoi', 'total_defect_qty', 'pi_time', 'pi_type'])

In [3]:
df.columns

Index(['sheet_id_chip_id', 'test_time', 'model_no', 'op_id', 'abbr_cat',
       'recipe_id', 'cassette_id', 'aoi', 'total_defect_qty',
       'defect_size_o_qty', 'defect_size_l_qty', 'defect_size_m_qty',
       'defect_size_s_qty', 'line_id', 'pi_time', 'pi_hour', 'pi_type'],
      dtype='object')

In [2]:
db = MySQLConnet('cim_piaoi')
#db.list_tables()
df = db.get_table('cim_pi_glass_202606')
df.head()

,sheet_id_chip_id,test_time,model_no,op_id,abbr_cat,recipe_id,cassette_id,aoi,total_defect_qty,defect_size_o_qty,defect_size_l_qty,defect_size_m_qty,defect_size_s_qty,line_id,pi_time,pi_hour,pi_type
0,AL5H151Z3K,2026-05-31 19:09:01,G215HVN01_VG,PI AOI,CF,608,CA0376,CAPIT203,172,0,3,16,153,CAPIC200,2026-06-01 00:33:11,2026-06-01 00:00:00,BPI
1,AL5H151Z3G,2026-05-31 19:13:11,G215HVN01_VG,PI AOI,CF,608,CA0376,CAPIT203,167,0,1,8,158,CAPIC200,2026-06-01 00:33:11,2026-06-01 00:00:00,BPI
2,AL5H151Z3M,2026-05-31 19:17:19,G215HVN01_VG,PI AOI,CF,608,CA0376,CAPIT203,155,0,4,12,139,CAPIC200,2026-06-01 00:33:11,2026-06-01 00:00:00,BPI
3,AL5H151Z3A,2026-05-31 19:21:22,G215HVN01_VG,PI AOI,CF,608,CA0376,CAPIT203,181,1,11,23,146,CAPIC200,2026-06-01 00:33:11,2026-06-01 00:00:00,BPI
4,YH6A5U84G,2026-06-01 00:35:12,B160UAV01,PCS1,TFT,2283,CA0364,CAAOI202,23,0,0,7,16,CAPIC400,2026-06-01 00:13:18,2026-05-31 23:00:00,API


In [15]:
from .. density.API_Config import API_Config
def _recipe_family(recipe_id: object) -> str:
    """
    依 recipe_id 粗分頁籤用途。
    """
    s = "" if recipe_id is None else str(recipe_id).strip()

    if len(s) == 4:
        if s.startswith("2"):
            return "UPI"
        if s.startswith("0"):
            return "PISpot_SPS"

    if len(s) == 3:
        return "ALL"

    return ""


def _add_router_total_density(clean_df: pd.DataFrame, api_cfg: API_Config) -> pd.DataFrame:
    """
    在 router 端重算 Total defect count / Total glass count / Total density。

    計算語意：
      1. 後端 density_summary 是 per recipe_id + per adc_def_code。
      2. 先在 recipe 粒度上去除同 recipe 多 defect code 造成的 glass_cnt 重複。
      3. 再依 tab family 聚合 total。
      4. merge 回原本每一列，讓前端直接取用。
    """
    if clean_df is None or clean_df.empty:
        clean_df = clean_df.copy()
        clean_df["total_defect_cnt"] = 0
        clean_df["total_glass_cnt"] = 0
        clean_df["total_density"] = 0.0
        return clean_df

    df = clean_df.copy()

    base_keys = ["pi_hour", "line_id", "aoi", "model", "glass_type"]
    recipe_keys = base_keys + ["recipe_id"]

    need_cols = set(base_keys + ["recipe_id", "adc_def_code", "glass_cnt", "defect_cnt"])
    miss = [c for c in need_cols if c not in df.columns]
    if miss:
        df["total_defect_cnt"] = 0
        df["total_glass_cnt"] = 0
        df["total_density"] = 0.0
        return df

    df["recipe_family"] = df["recipe_id"].apply(_recipe_family)

    df["glass_cnt"] = pd.to_numeric(df["glass_cnt"], errors="coerce").fillna(0)
    df["defect_cnt"] = pd.to_numeric(df["defect_cnt"], errors="coerce").fillna(0)

    # per recipe 去重 glass_cnt，避免同一 recipe 下多 defect code 重複加總 glass_cnt
    recipe_base = (
        df.groupby(recipe_keys + ["recipe_family"], dropna=False)
          .agg(
              recipe_total_glass_cnt=("glass_cnt", "max"),
              recipe_total_defect_cnt=("defect_cnt", "sum"),
          )
          .reset_index()
    )

    # UPI
    upi_base = recipe_base[recipe_base["recipe_family"].isin(["UPI", "ALL"])].copy()
    upi_total = (
        upi_base.groupby(base_keys, dropna=False)
                .agg(
                    total_glass_cnt=("recipe_total_glass_cnt", "sum"),
                    total_defect_cnt=("recipe_total_defect_cnt", "sum"),
                )
                .reset_index()
    )
    upi_total["recipe_family"] = "UPI"

    # PISpot / SPS：目前你的 recipe 規則兩者共用 0xxx + 3碼
    pis_sps_base = recipe_base[recipe_base["recipe_family"].isin(["PISpot_SPS", "ALL"])].copy()
    pis_sps_total = (
        pis_sps_base.groupby(base_keys, dropna=False)
                    .agg(
                        total_glass_cnt=("recipe_total_glass_cnt", "sum"),
                        total_defect_cnt=("recipe_total_defect_cnt", "sum"),
                    )
                    .reset_index()
    )
    pis_sps_total["recipe_family"] = "PISpot_SPS"

    total_df = pd.concat([upi_total, pis_sps_total], ignore_index=True)

    if total_df.empty:
        df["total_defect_cnt"] = 0
        df["total_glass_cnt"] = 0
        df["total_density"] = 0.0
        return df.drop(columns=["recipe_family"], errors="ignore")

    total_df["total_density"] = (
        total_df["total_defect_cnt"] /
        total_df["total_glass_cnt"].replace(0, pd.NA)
    ).fillna(0).round(3)

    df = df.merge(
        total_df,
        on=base_keys + ["recipe_family"],
        how="left"
    )

    df["total_defect_cnt"] = df["total_defect_cnt"].fillna(0)
    df["total_glass_cnt"] = df["total_glass_cnt"].fillna(0)
    df["total_density"] = df["total_density"].fillna(0.0)

    return df.drop(columns=["recipe_family"], errors="ignore")


ImportError: attempted relative import with no known parent package

In [ ]:
db = MySQLConnet('piaoi_density')
#db.list_tables()
#df = db.get_table('rtms_aoi300_raw_202605')
tbn = 'density_code_summary_202605'
#df = db.get_runs_between(tbn,  '2026-05-02', '2026-05-05', time_col="pi_hour")
df = db.get_rows(tbn, {'line_id': 'CAPIC400',  'model': 'B160UAV01', 'adc_def_code':'NPI_TFT'})
pd.DataFrame(df)#['adc_def_code'].unique()

,line_id,aoi,model,glass_type,pi_hour,recipe_id,adc_def_code,recipe_total_glass_cnt,recipe_total_defect_cnt,recipe_total_density,...,small_defect_count,middle_defect_count,large_defect_count,over_defect_count,glass,glass_size_detail,comment,action,Editor,modify_time
0,CAPIC400,aoi200,B160UAV01,TFT,2026-05-31 06:00:00,2283,NPI_TFT,3,92,30.667,...,1,4,0,0,"YH6A5VF1G,YH6A5VF1H,YH6A5VF1J","{""YH6A5VF1G"": {""S"": 0, ""M"": 1, ""L"": 0, ""O"": 0,...",,,,2026-06-01 07:05:17
1,CAPIC400,aoi200,B160UAV01,TFT,2026-05-31 06:00:00,0283,NPI_TFT,3,608,202.667,...,20,0,0,0,"YH6A5VF1G,YH6A5VF1H,YH6A5VF1J","{""YH6A5VF1G"": {""S"": 13, ""M"": 0, ""L"": 0, ""O"": 0...",,,,2026-06-01 07:05:17
2,CAPIC400,aoi200,B160UAV01,TFT,2026-05-31 13:00:00,4283,NPI_TFT,7,1614,230.571,...,46,21,3,0,"YH6A5U88A,YH6A5U88B,YH6A5U88C,YH6A5U88D,YH6A5U...","{""YH6A5U88A"": {""S"": 3, ""M"": 7, ""L"": 0, ""O"": 0,...",,,,2026-06-01 13:35:13
3,CAPIC400,aoi200,B160UAV01,TFT,2026-05-31 16:00:00,2283,NPI_TFT,7,209,29.857,...,10,11,0,0,"YH6A5VF7A,YH6A5VF7B,YH6A5VF7C,YH6A5VF7D,YH6A5V...","{""YH6A5VF7A"": {""S"": 1, ""M"": 1, ""L"": 0, ""O"": 0,...",,,,2026-06-01 16:45:17
4,CAPIC400,aoi200,B160UAV01,TFT,2026-05-31 16:00:00,0283,NPI_TFT,7,902,128.857,...,59,4,0,0,"YH6A5VF7A,YH6A5VF7B,YH6A5VF7C,YH6A5VF7D,YH6A5V...","{""YH6A5VF7A"": {""S"": 7, ""M"": 0, ""L"": 0, ""O"": 0,...",,,,2026-06-01 16:45:17
5,CAPIC400,aoi200,B160UAV01,TFT,2026-05-31 20:00:00,2283,NPI_TFT,7,162,23.143,...,11,4,0,0,"YH6A5UA1A,YH6A5UA1B,YH6A5UA1C,YH6A5UA1D,YH6A5U...","{""YH6A5UA1A"": {""S"": 0, ""M"": 0, ""L"": 0, ""O"": 0,...",,,,2026-06-01 20:45:09
6,CAPIC400,aoi200,B160UAV01,TFT,2026-05-31 20:00:00,0283,NPI_TFT,7,1169,167.000,...,76,5,0,0,"YH6A5UA1A,YH6A5UA1B,YH6A5UA1C,YH6A5UA1D,YH6A5U...","{""YH6A5UA1A"": {""S"": 0, ""M"": 0, ""L"": 0, ""O"": 0,...",,,,2026-06-01 20:45:09


In [ ]:
df[(df['aoi'] =='aoi100' )& (df['model'] == 'G215HVN01') ] # & (df['pi_hour'] == '2026-05-05 10:00:00')

,line_id,aoi,model,glass_type,recipe_id,pi_hour,adc_def_code,defect_cnt,glass_cnt,density,...,middle_defect_count,large_defect_count,over_defect_count,glass,glass_size_detail,shift,comment,action,Editor,modify_time


In [ ]:
db = MySQLConnet('rtms_piaoi_other')
db.list_tables()
df = db.get_table('rtms_aoi300_raw_202605')

#df['defect_size'].unique()

2026-05-05 11:46:03,167 - INFO - [list_tables] 成功取得資料表名稱，共 4 張表。


In [ ]:
rows = df[df['sheet_id_chip_id'] =='ZG6A5N17C']
print(len(rows))
rows['test_time'].unique()
rows.iloc[0].to_dict()

191


{'id': 4964,
 'sheet_id_chip_id': 'ZG6A5N17C',
 'chip_id': 'ZG6A5N17C2',
 'test_time': Timestamp('2026-05-04 16:21:16'),
 'defect_size': 'S',
 'size_class': '6',
 'pox_x1': 310572,
 'pox_y1': 1035260,
 'adc_def_code': '',
 'retype_def_code': '',
 'image_file_name': 'CAAOI300.M270QAN06.M270QAN06-T-API-260225.CA020601.ZG6A5N17C.1.bt5.1777882876.jpg',
 'img_file_url_path': 'PIT/2605/04/CAAOI300/ZG6A5N17C/1621/',
 'pic_path': 'http://10.97.139.98:1454/CAAOI300/CA020601/ZG6A5N17C/PCS1/58703/CaptureImage/small/CAAOI300.M270QAN06.M270QAN06-T-API-260225.CA020601.ZG6A5N17C.1.bt5.1777882876.jpg',
 'recipe_id': 'M270QAN06-T-API-260225',
 'line_id': 'Null',
 'aoi': 'aoi300',
 'model': 'M270QAN06',
 'glass_type': 'TFT',
 'pi_time': NaT,
 'pi_type': 'API',
 'cst_id': 'CA020601',
 'defect_count': 190,
 'defect_id': '1',
 'source_file': 'CAAOI300.M270QAN06.M270QAN06-T-API-260225.CA020601.ZG6A5N17C.58703.defect',
 'source_mtime': Timestamp('2026-05-04 16:23:27'),
 'update_time': Timestamp('2026-05-05 0

In [ ]:
df['recipe_id'].unique()

array(['M270DAN7F-T-API', 'M270QAN06-T-API-260225',
       'B160UAN4A-T-API-20260203', 'B180UAN01-T-API-260405',
       'M270QAN07-T-API_260410', 'M340QAR1B-T-API-060206',
       'M270TAN2Z-T-API-260416', 'M270HAN03-T-API', 'M270HAN1C-T-API',
       'B140UAN04-T-API', 'CELL-ITO', 'M215HAN01-T-API-060204',
       'B160UAN4A-T-BPI'], dtype=object)

In [ ]:
cols = ['aoi', 'test_time', 'glass_type', 'glass_type', 'defect_count'] #same gld
for (gld,  ) , rows in df.groupby(['sheet_id_chip_id']):
    print(gld, len(rows))
    print(rows[cols])


MQ6A5TK12 4
        aoi           test_time glass_type glass_type  defect_count
368  aoi300 2026-05-02 08:30:45        ITO        ITO             0
369  aoi300 2026-05-02 08:30:45        ITO        ITO             0
370  aoi300 2026-05-02 08:30:45        ITO        ITO             0
371  aoi300 2026-05-02 08:30:45        ITO        ITO             0
MQ6A5TK13 4
        aoi           test_time glass_type glass_type  defect_count
372  aoi300 2026-05-02 08:26:03        ITO        ITO             0
373  aoi300 2026-05-02 08:26:03        ITO        ITO             0
374  aoi300 2026-05-02 08:26:03        ITO        ITO             0
375  aoi300 2026-05-02 08:26:03        ITO        ITO             0
MQ6A5TK14 4
        aoi           test_time glass_type glass_type  defect_count
376  aoi300 2026-05-02 08:21:10        ITO        ITO             0
377  aoi300 2026-05-02 08:21:10        ITO        ITO             0
378  aoi300 2026-05-02 08:21:10        ITO        ITO             0
379  aoi300 

In [ ]:
db.get_rows('rtms_aoi300_raw_202604',{'defect_size': 'OK'})

[{'id': 17860074,
  'sheet_id_chip_id': 'MQ6A5PN71',
  'chip_id': 'MQ6A5PN710',
  'test_time': datetime.datetime(2026, 4, 24, 13, 7, 26),
  'defect_size': 'OK',
  'size_class': '0',
  'pox_x1': 0,
  'pox_y1': 0,
  'adc_def_code': '',
  'retype_def_code': '',
  'image_file_name': 'CAAOI300.CELL-ITO.CELL-ITO.CA002001.MQ6A5PN71.GlassImage_modality5.bm.1777007246.jpg',
  'img_file_url_path': 'PIT/2604/24/CAAOI300/MQ6A5PN71/1307/Map/',
  'pic_path': 'http://10.97.139.98:1454/CAAOI300/CA002001/MQ6A5PN71/PCS1/47086/Map/CAAOI300.CELL-ITO.CELL-ITO.CA002001.MQ6A5PN71.GlassImage_modality5.bm.1777007246.jpg',
  'recipe_id': 'CELL-ITO',
  'line_id': 'Null',
  'aoi': 'aoi300',
  'model': 'CELL-ITO',
  'glass_type': 'ITO',
  'pi_time': None,
  'pi_type': 'CELL-ITO',
  'cst_id': 'CA002001',
  'defect_count': 0,
  'defect_id': 'MACRO_1',
  'source_file': 'CAAOI300.CELL-ITO.CELL-ITO.CA002001.MQ6A5PN71.47086.defect',
  'source_mtime': datetime.datetime(2026, 4, 24, 13, 9, 27),
  'update_time': datetime.d

In [17]:
import csv
df = pd.read_csv('D:/A0_Project/PI_SYSTEM/models/density/AOI_Spec2.csv',encoding='cp950')
df

,序號,Model,產品別,Defect Type,Item1,Defect Size,OOC,OOS
0,1,B070AAN01,Normal,Polymer,BIG,OLMS,5.0,10.0
1,2,B070AAN1Y,Normal,Polymer,BIG,OLMS,5.0,10.0
2,3,B070VTN01,Normal,Polymer,BIG,OLMS,5.0,10.0
3,4,B080EAN05,Normal,Polymer,BIG,OLMS,5.0,10.0
4,5,B101EAN2A,Normal,Polymer,BIG,OLMS,5.0,10.0
...,...,...,...,...,...,...,...,...
945,159,P270QAN01,Normal,NPI_TFT,BIG,OLMS,8.0,10.0
946,160,P270QAN02,Normal,NPI_TFT,BIG,OLMS,8.0,10.0
947,161,P281SAN01,Normal,NPI_TFT,BIG,OLMS,8.0,10.0
948,162,Y160UAN03,Normal,NPI_TFT,BIG,OLMS,8.0,10.0


In [22]:
df.rename({
    'Model':'model',
    '產品別':'MODEL_TYPE',
    'Defect Type':'adc_def_code',
    'Item1':'PROCESS_TYPE',
    'Defect Size':'defect_size',
    'OOC':'OOC',
    'OOS':'OOS'

},axis=1, inplace=True)

df.loc[df.index, ['line_id', 'glass_type', 'Editor', 'modify_time']] = ''
df.loc[df.index, ['drop']] = 'F'
df = df[[cl for cl in df.columns if cl in spec_df.columns]]
df.head()

,model,MODEL_TYPE,adc_def_code,PROCESS_TYPE,defect_size,OOC,OOS,line_id,glass_type,Editor,modify_time,drop
0,B070AAN01,Normal,Polymer,BIG,OLMS,5.0,10.0,,,,,F
1,B070AAN1Y,Normal,Polymer,BIG,OLMS,5.0,10.0,,,,,F
2,B070VTN01,Normal,Polymer,BIG,OLMS,5.0,10.0,,,,,F
3,B080EAN05,Normal,Polymer,BIG,OLMS,5.0,10.0,,,,,F
4,B101EAN2A,Normal,Polymer,BIG,OLMS,5.0,10.0,,,,,F


In [25]:
type(df['OOC'][0])

numpy.float64

In [28]:
new = pd.DataFrame()

for line_id in spec_df['line_id'].unique():
    print(line_id)
    df['line_id'] = line_id
    new = pd.concat([new, df.copy()])
    print(len(new))

spec2 = pd.DataFrame()
for t in spec_df['glass_type'].unique():
    new['glass_type'] = t
    spec2 = pd.concat([spec2, new.copy()])

CAPIC100
950
CAPIC200
1900
CAPIC300
2850
CAPIC400
3800
CAPIC500
4750
CAPIC600
5700
CAPIC700
6650


C:\Users\Ray\AppData\Local\Temp\ipykernel_16356\1485156070.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['line_id'] = line_id


In [29]:
db.save_table('default_spec_table', spec2)

2026-06-23 14:58:41,564 - INFO - [save_table] piaoi_density.default_spec_table 已寫入 13300 列（去除 0 重複）


13300

In [ ]:
http://tcweb002.corpnet.auo.com/fafle001/image/PS/4/AR0DJ600R4/RAR0DJ600R4_0038_001_1011544_1567595.jpg
http://tcweb002.corpnet.auo.com/fafle001/image/PS/4/AR0DJ600R4/RAR0DJ600R4_0010_001_1341804_1290293.jpg

In [21]:
[cl for cl in spec_df.columns if cl not in df.columns]

['line_id', 'glass_type', 'Editor', 'modify_time', 'drop']

In [6]:
db = MySQLConnet('piaoi_density')
spec_df = db.get_table('default_spec_table')
#cols = ['model']
print(len(spec_df))
spec_df


7604


,line_id,model,glass_type,adc_def_code,defect_size,OOC,OOS,Editor,modify_time,drop,MODEL_TYPE,PROCESS_TYPE
0,CAPIC100,B070AAN01,TFT,PIS With Particle,OL,0.28,0.33,預設,2025-12-24 15:56:23,F,Normal,BIG
1,CAPIC100,B070AAN01,TFT,PIS With Particle,OLMS,2,3,預設,2026-06-18 10:46:24,T,Normal,BIG
2,CAPIC100,B070AAN01,TFT,PI_Spot_NP,OL,0.33,0.57,預設,2025-12-09 08:00:00,F,Normal,BIG
3,CAPIC100,B070AAN01,TFT,PI_Spot_NP,OLMS,300,300,預設,2026-06-18 10:55:43,T,Normal,BIG
4,CAPIC100,B070AAN01,TFT,Polymer,OLMS,5,10,預設,2025-12-09 08:00:00,F,Normal,BIG
...,...,...,...,...,...,...,...,...,...,...,...,...
7599,CAPIC700,M270QAN06,TFT,NPI_TFT,OLMS,8,10,預設,2026-06-18 15:53:22,F,Normal,BIG
7600,CAPIC100,TEST,TFT,NPI_TFT,OLMS,0.5,0.6,預設,2026-06-22 08:41:35,F,Normal,BIG
7601,CAPIC400,B160UAV01,TFT,Polymer,OLMS,5,10,預設,2026-06-23 07:48:29,F,高階,BIG
7602,CAPIC400,B160UAV01,TFT,SSIU_Polymer,OLMS,40,80,預設,2026-06-23 07:49:03,F,高階,BIG


In [27]:
spec_df['glass_type'].unique()

array(['TFT', 'CF'], dtype=object)

In [ ]:
import pandas as pd


def build_expand_spec_df(
    model_list,
    line_id_list=None,
    glass_type_list=None,
    size_group_list=None,
    default_ooc=150,
    default_oos=200,
    editor="預設",
    modify_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
):
    #if line_id_list is None:
    #    line_id_list = ["ALL"]

    if glass_type_list is None:
        glass_type_list = ["TFT", "CF"]

    if size_group_list is None:
        size_group_list = ["S", "MS", "LMS", "O", "OL", "OLM", "OLMS"]

    def clean_list(values):
        out = []
        for v in values or []:
            if v is None:
                continue
            s = str(v).strip()
            if not s:
                continue
            if s.lower() in {"nan", "none", "null", "<na>", "nat"}:
                continue
            out.append(s)
        return sorted(set(out))

    models = clean_list(model_list)
    #lines = clean_list(line_id_list)
    glass_types = clean_list(glass_type_list)
    size_groups = clean_list(size_group_list)

    rows = []

    #for line_id in lines:
    for model in models:
        for glass_type in glass_types:
            for size_group in size_groups:
                rows.append({
                    "model": model,
                    "glass_type": glass_type,
                    "defect_size": size_group,
                    "OOC": default_ooc,
                    "OOS": default_oos,
                    "Editor": editor,
                    "modify_time": modify_time ,
                    "drop" : 'F'
                })

    return pd.DataFrame(rows)

spec_df = build_expand_spec_df([v for v in models if 'test' not in v])
spec_df

,model,glass_type,defect_size,OOC,OOS,Editor,modify_time,drop
0,B070AAN01,CF,LMS,150,200,預設,2026-04-28 13:57:13,F
1,B070AAN01,CF,MS,150,200,預設,2026-04-28 13:57:13,F
2,B070AAN01,CF,O,150,200,預設,2026-04-28 13:57:13,F
3,B070AAN01,CF,OL,150,200,預設,2026-04-28 13:57:13,F
4,B070AAN01,CF,OLM,150,200,預設,2026-04-28 13:57:13,F
...,...,...,...,...,...,...,...,...
2585,Z335XAN01_TEST,TFT,O,150,200,預設,2026-04-28 13:57:13,F
2586,Z335XAN01_TEST,TFT,OL,150,200,預設,2026-04-28 13:57:13,F
2587,Z335XAN01_TEST,TFT,OLM,150,200,預設,2026-04-28 13:57:13,F
2588,Z335XAN01_TEST,TFT,OLMS,150,200,預設,2026-04-28 13:57:13,F


In [ ]:
spec_df.columns

Index(['model', 'glass_type', 'defect_size', 'OOC', 'OOS', 'Editor',
       'modify_time', 'drop'],
      dtype='object')

In [ ]:
db = MySQLConnet('piaoi_bpi_density')
#db.list_tables()
#db.save_table('default_spec_table', spec_df)
db.get_rows('default_spec_table', {'model': 'test_0429'})

[{'model': 'test_0429',
  'glass_type': 'TFT',
  'defect_size': 'MS',
  'OOC': 111,
  'OOS': 222,
  'Editor': '預設',
  'modify_time': '2026-04-29 10:33:03',
  'drop': 'F'}]

In [ ]:
df = db.get_runs_between('bpi_api_summary_202604', '2026-04-26', '2026-04-28', 'scan_hour')
df.iloc[-1,:].to_dict()

{'id': 3,
 'aoi': 'aoi200',
 'model': 'B160UAN05',
 'scan_hour': Timestamp('2026-04-26 11:00:00'),
 'cassette_id': 'AA1659',
 'glass_side': 'TFT',
 'recipe_id': '4258',
 'pi_type': 'BPI',
 'run_day': datetime.date(2026, 4, 26),
 'glass_count': 1,
 'total_defect_count': 4851,
 'small_defect_count': 473,
 'middle_defect_count': 1563,
 'large_defect_count': 1359,
 'over_defect_count': 0,
 'density': 4851.0,
 'glass_list': 'VN6A3VE6A',
 'glass_size_detail': '{"VN6A3VE6A": {"S": 473, "M": 1563, "L": 1359, "O": 0, "T": 3395, "line_id": "CAPIC200", "test_time": "2026-04-26 12:09:45"}}',
 'source_db': 'cim_piaoi',
 'source_table': 'cim_pi_glass_202604',
 'comment': '',
 'action': '',
 'editor': '',
 'modify_time': None,
 'update_time': Timestamp('2026-04-28 11:08:34')}

In [ ]:
db = MySQLConnet('cim_piaoi')
df = db.get_runs_between('cim_pi_glass_202604', '2026-04-26', '2026-04-27', 'pi_hour')
df.columns

Index(['sheet_id_chip_id', 'test_time', 'model_no', 'op_id', 'abbr_cat',
       'recipe_id', 'cassette_id', 'aoi', 'total_defect_qty',
       'defect_size_o_qty', 'defect_size_l_qty', 'defect_size_m_qty',
       'defect_size_s_qty', 'line_id', 'pi_time', 'pi_hour', 'pi_type'],
      dtype='object')

In [ ]:
rdf = df[(df['cassette_id'] == 'AA1659') & (df['model_no'] == 'B160UAN05') ] #(df['model'] == 'B160UAN05') & (df['scan_hour'] == "2026-04-26 11:00:00") 
rdf

,sheet_id_chip_id,test_time,model_no,op_id,abbr_cat,recipe_id,cassette_id,aoi,total_defect_qty,defect_size_o_qty,defect_size_l_qty,defect_size_m_qty,defect_size_s_qty,line_id,pi_time,pi_hour,pi_type
89,VN6A3VE6A,2026-04-26 12:09:45,B160UAN05,PCS1,TFT,4258,AA1659,CAAOI202,4851,0,1359,1563,473,CAPIC200,2026-04-26 18:21:21,2026-04-26 17:00:00,BPI


In [ ]:
730*0.65-(120+22+13)
#100-3-(4*1.5)

319.5

In [ ]:
rdf = df[(df['cassette_id'] == "AA1659") ] #(df['model'] == 'B160UAN05') & (df['scan_hour'] == "2026-04-26 11:00:00") 

for (recipe, glass_type, code, ), rows in rdf.groupby(['cassette_id','recipe_id',  'glass_side']):
    print(recipe, glass_type,  code, len(rows))
    print(rows[['glass_count', 'total_defect_count',
       'small_defect_count', 'middle_defect_count', 'large_defect_count',
       'over_defect_count', 'density']])

KeyError: 'glass_side'

In [ ]:
473+1563+1359

3395

In [ ]:
#db = MySQLConnet('piaoi_capa')
db = MySQLConnet('piaoi_bpi_same_point')
#for tbn in db.list_tables():#[v for v in db.list_tables() if 'aoi300' not in v]:
    #db.drop_table(tbn)

"""
db.save_table("default_spec_table", pd.DataFrame(columns=[
            "model",
            "glass_side",
            "defect_size",
            "OOC",
            "OOS",
            "editor",
            "modify_time",
            "drop",
        ]))
"""
df = db.get_table("bpi_same_point_offset_summary_202605")
df.iloc[0,:].to_dict()

{'id': 281,
 'model': 'B153UAN03',
 'glass_side': 'TFT',
 'glass_id': 'Y26A5RV7A',
 'scan_hour': Timestamp('2026-05-14 18:00:00'),
 'run_day': Timestamp('2026-05-14 00:00:00'),
 'tab': 'PISpot',
 'bpi_aoi': 'aoi200',
 'bpi_scan_time': Timestamp('2026-05-12 12:08:06'),
 'bpi_recipe_id': '4295',
 'api_aoi': 'aoi200',
 'api_scan_time': Timestamp('2026-05-14 19:13:06'),
 'api_recipe_id': '0295',
 'offset_um': 5,
 'bpi_defect_count': 70,
 'api_defect_count': 246,
 'matched_pair_count': 0,
 'matched_bpi_defect_count': 0,
 'matched_api_defect_count': 0,
 'unmatched_bpi_defect_count': 70,
 'unmatched_api_defect_count': 246,
 'matched_bpi_s_count': 0,
 'matched_bpi_m_count': 0,
 'matched_bpi_l_count': 0,
 'matched_bpi_o_count': 0,
 'matched_api_s_count': 0,
 'matched_api_m_count': 0,
 'matched_api_l_count': 0,
 'matched_api_o_count': 0,
 'matched_size_transition_json': '{}',
 'gen_time': Timestamp('2026-05-26 11:25:41')}

In [ ]:
df = db.get_table("bpi_same_point_202605")
df[df['model'] == "G215HVN01"]

,id,model,glass_side,glass_id,scan_hour,run_day,tab,bpi_aoi,bpi_line_id,bpi_recipe_id,...,api_over_defect_count,pair_status,pair_message,default_offset_um,matched_points_json,comment,action,editor,modify_time,gen_time
26,69,G215HVN01,CF,AL5H151YDN,2026-05-05 23:00:00,2026-05-05,aoi100,aoi200,CAPIC200,5243,...,0,OK,,20,[],,,,NaT,2026-05-26 11:25:30
27,70,G215HVN01,CF,AL5H151YEA,2026-05-05 23:00:00,2026-05-05,aoi100,aoi200,CAPIC200,5243,...,0,OK,,20,[],,,,NaT,2026-05-26 11:25:30
28,71,G215HVN01,CF,AL5H151YEK,2026-05-05 23:00:00,2026-05-05,aoi100,aoi200,CAPIC200,5243,...,0,OK,,20,[],,,,NaT,2026-05-26 11:25:30
29,72,G215HVN01,CF,AL5H151YEP,2026-05-05 23:00:00,2026-05-05,aoi100,aoi200,CAPIC200,5243,...,0,OK,,20,[],,,,NaT,2026-05-26 11:25:30
30,73,G215HVN01,CF,AL5H151YEQ,2026-05-05 23:00:00,2026-05-05,aoi100,aoi200,CAPIC200,5243,...,0,OK,,20,[],,,,NaT,2026-05-26 11:25:30
31,74,G215HVN01,CF,AL5H151YES,2026-05-05 23:00:00,2026-05-05,aoi100,aoi200,CAPIC200,5243,...,0,OK,,20,[],,,,NaT,2026-05-26 11:25:30
32,75,G215HVN01,CF,AL5H151YEU,2026-05-05 23:00:00,2026-05-05,aoi100,aoi200,CAPIC200,5243,...,0,OK,,20,"[{""offset_um"": 20, ""distance"": 19.849433241279...",,,,NaT,2026-05-26 11:25:30
33,76,G215HVN01,CF,AL5H151YK5,2026-05-05 23:00:00,2026-05-05,aoi100,aoi200,CAPIC200,5243,...,0,OK,,20,"[{""offset_um"": 20, ""distance"": 16.278820596099...",,,,NaT,2026-05-26 11:25:30


In [ ]:
df = db.get_table("default_spec_table")
df.iloc[-1,:].to_dict()

{'model': 'nan',
 'glass_side': 'nan',
 'defect_size': 'nan',
 'OOC': '5',
 'OOS': '10',
 'editor': '預設',
 'modify_time': '2026-05-27 15:52:23',
 'drop': 'F'}

In [ ]:
[{"offset_um": 20, 
  "distance": 2.0, 
  "dx": -2.0, "dy": 0.0, 
  "match_rank": 1, 
  "bpi_defect_uid": "RTMS|47373", 
  "bpi_chip_id": "YL6A5MX7D26", 
  "bpi_x": 1425267.0, "bpi_y": 940299.0,
  "bpi_defect_size": "S", 
  "bpi_adc_def_code": "", 
  "bpi_retype_code": "", 
  "bpi_pic_path": "http://10.97.139.98:1454/CAAOI300/AA001901/YL6A5MX7D/PCS1/61210/CaptureImage/small/CAAOI300.B156HAN2U.B156HAN2U-T-BPI.AA001901.YL6A5MX7D.61.bt5.1777971783.jpg", "bpi_pic_name": "CAAOI300.B156HAN2U.B156HAN2U-T-BPI.AA001901.YL6A5MX7D.61.bt5.1777971783.jpg", 
  "api_defect_uid": "CIM|YL6A5MX7D|2026-05-25 08:30:57|1|1425269|940299|RV1_1425269_940299_19.jpg", 
  "api_chip_id": "1", 
  "api_x": 1425269.0, "api_y": 940299.0, 
  "api_defect_size": "M", 
  "api_adc_def_code": "NPI_OIL", 
  "api_retype_code": "", 
  "api_pic_path": "http://l6apaimg103/dms/CELAIDI_L6A/PIT/2605/25/CAAOI202/YL6A5MX7D/0830/", 
  "api_pic_name": "RV1_1425269_940299_19.jpg"},
    {"offset_um": 20, "distance": 18.439088914585774, "dx": -12.0, "dy": 14.0, "match_rank": 2, "bpi_defect_uid": "RTMS|47371", "bpi_chip_id": "YL6A5MX7D24", "bpi_x": 1374237.0, "bpi_y": 248056.0, "bpi_defect_size": "S", "bpi_adc_def_code": "", "bpi_retype_code": "", "bpi_pic_path": "http://10.97.139.98:1454/CAAOI300/AA001901/YL6A5MX7D/PCS1/61210/CaptureImage/small/CAAOI300.B156HAN2U.B156HAN2U-T-BPI.AA001901.YL6A5MX7D.59.bt5.1777971783.jpg", "bpi_pic_name": "CAAOI300.B156HAN2U.B156HAN2U-T-BPI.AA001901.YL6A5MX7D.59.bt5.1777971783.jpg", "api_defect_uid": "CIM|YL6A5MX7D|2026-05-25 08:30:57|Z|1374249|248042|RV1_1374249_248042_44.jpg", "api_chip_id": "Z", "api_x": 1374249.0, "api_y": 248042.0, "api_defect_size": "L", "api_adc_def_code": "OP_Defect", "api_retype_code": "", "api_pic_path": "http://l6apaimg103/dms/CELAIDI_L6A/PIT/2605/25/CAAOI202/YL6A5MX7D/0830/",
      "api_pic_name": "RV1_1374249_248042_44.jpg"}]

In [4]:
7.8*1.25

9.75

In [7]:
db = MySQLConnet('cim_cell_aoi_to_array')
tbns = db.list_tables()
tbns

2026-06-26 15:12:27,766 - INFO - [list_tables] 成功取得資料表名稱，共 12 張表。


['api_aoi_summary_202606',
 'incoming_glass_summary_202606',
 'incoming_governance_state',
 'incoming_same_point_detail_202606',
 'incoming_source_array_mor_defect_raw_202606',
 'incoming_source_array_tar_defect_raw_202605',
 'incoming_source_array_tar_defect_raw_202606',
 'incoming_source_array_tos_defect_raw_202604',
 'incoming_source_array_tos_defect_raw_202605',
 'incoming_source_array_tos_defect_raw_202606',
 'incoming_source_cf_oc_defect_raw_202606',
 'incoming_source_cf_ps_defect_raw_202606']

In [37]:
rows = db.get_rows('incoming_source_cf_ps_defect_raw_202606', {'sheet_id':'AR0DJ600R4'})
#rows.sort_values(by='scan_time', ascending=True, inplace=True)
rows

[{'id': 5,
  'sheet_id': 'AR0DJ600R4',
  'chip_id': 'AR0DJ600R4E',
  'model_no': 'M270DAN7J-VC51MP000',
  'scan_time': datetime.datetime(2026, 6, 22, 4, 50, 33),
  'eqp_id': 'FASAOI30',
  'op': 'PS',
  'repair_time': datetime.datetime(2026, 6, 22, 9, 31, 36),
  'repair_code': 'nan',
  'repair_eqp_id': 'FAAREP42',
  'repair_op': 'REPAIR-PS',
  'op_id': 'PS',
  'defect_no': '0005',
  'defect_code': 'RB',
  'defect_size': 'M',
  'defect_size_raw': 'M',
  'ori_x': 1430703.0,
  'ori_y': 1190736.0,
  'trans_x': 1190736.0,
  'trans_y': 69297.0,
  'image_name': 'RAR0DJ600R4_0005_001_1430703_1190736.jpg',
  'img_url_path': 'http://tcweb002.corpnet.auo.com/fafle001/image/PS/4/AR0DJ600R4/RAR0DJ600R4_0005_001_1430703_1190736.jpg',
  'source_group_key': 'CF|AR0DJ600R4|PS|2026-06-22 04:50:33',
  'source_defect_uid': 'CF|AR0DJ600R4|PS|2026-06-22 04:50:33|AR0DJ600R4E|0005|RB|M|1430703|1190736|RAR0DJ600R4_0005_001_1430703_1190736.jpg',
  'raw_json': '{"glass_id": "AR0DJ600R4", "chip_id": "AR0DJ600R4E",

In [21]:
#rows = db.get_runs_delta_days('incoming_glass_summary_202606', 3, 'scan_time')
rows['sheet_id'].iloc[0:50].to_csv('test_sheet.csv', index=False, header=False)

In [8]:

for tbn in db.list_tables():
    db.drop_table(tbn)


2026-06-26 15:12:31,387 - INFO - [list_tables] 成功取得資料表名稱，共 12 張表。


2026-06-26 15:12:32,830 - INFO - [drop_table] 資料表 'api_aoi_summary_202606' 已刪除.
2026-06-26 15:12:33,285 - INFO - [drop_table] 資料表 'incoming_glass_summary_202606' 已刪除.
2026-06-26 15:12:33,657 - INFO - [drop_table] 資料表 'incoming_governance_state' 已刪除.
2026-06-26 15:12:34,317 - INFO - [drop_table] 資料表 'incoming_same_point_detail_202606' 已刪除.
2026-06-26 15:12:34,643 - INFO - [drop_table] 資料表 'incoming_source_array_mor_defect_raw_202606' 已刪除.
2026-06-26 15:12:35,181 - INFO - [drop_table] 資料表 'incoming_source_array_tar_defect_raw_202605' 已刪除.
2026-06-26 15:12:35,614 - INFO - [drop_table] 資料表 'incoming_source_array_tar_defect_raw_202606' 已刪除.
2026-06-26 15:12:36,222 - INFO - [drop_table] 資料表 'incoming_source_array_tos_defect_raw_202604' 已刪除.
2026-06-26 15:12:36,539 - INFO - [drop_table] 資料表 'incoming_source_array_tos_defect_raw_202605' 已刪除.
2026-06-26 15:12:37,127 - INFO - [drop_table] 資料表 'incoming_source_array_tos_defect_raw_202606' 已刪除.
2026-06-26 15:12:37,457 - INFO - [drop_table] 資料表 'in

In [9]:
rows = db.get_rows('incoming_same_point_detail_202606', {'cell_aoi':'CAAOI202'})
df = pd.DataFrame(rows)
df['match_status'].unique()

array(['NO_SAME_POINT', 'SOURCE_NOT_FOUND'], dtype=object)

In [5]:
df = db.get_table('incoming_same_point_detail_202606')
df

,id,sheet_id,scan_time,model_no,abbr_cat,process,recipe_id,cassette_id,cell_aoi,cell_line_id,...,source_scan_time,source_defect_cnt,same_point_offset,same_point_defect_cnt,same_point_rate,point_detail,match_status,match_status_detail,create_time,update_time
0,1,5T6A5604A,2026-06-10 08:14:44,B160UAN5L,TFT,ARRAY,0335,CA0175,CAAOI202,CAPIC100,...,2026-06-05 16:49:03,152.0,1000.0,0.0,0.000000,[],NO_SAME_POINT,source found but no same point; source_defect_...,2026-06-11 16:22:45,2026-06-11 16:22:45
1,2,5T6A5604A,2026-06-10 08:14:44,B160UAN5L,TFT,ARRAY,0335,CA0175,CAAOI202,CAPIC100,...,NaT,NaN,1000.0,NaN,NaN,[],SOURCE_NOT_FOUND,no source defect group for TAR,2026-06-11 16:22:45,2026-06-11 16:22:45
2,3,5T6A5604A,2026-06-10 08:14:44,B160UAN5L,TFT,ARRAY,0335,CA0175,CAAOI202,CAPIC100,...,2026-06-06 15:51:56,1.0,1000.0,0.0,0.000000,[],NO_SAME_POINT,source found but no same point; source_defect_...,2026-06-11 16:22:45,2026-06-11 16:22:45
3,4,5T6A5604B,2026-06-10 08:10:47,B160UAN5L,TFT,ARRAY,0335,CA0175,CAAOI202,CAPIC100,...,2026-06-05 16:49:03,177.0,1000.0,0.0,0.000000,[],NO_SAME_POINT,source found but no same point; source_defect_...,2026-06-11 16:22:45,2026-06-11 16:22:45
4,5,5T6A5604B,2026-06-10 08:10:47,B160UAN5L,TFT,ARRAY,0335,CA0175,CAAOI202,CAPIC100,...,NaT,NaN,1000.0,NaN,NaN,[],SOURCE_NOT_FOUND,no source defect group for TAR,2026-06-11 16:22:45,2026-06-11 16:22:45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
329,330,YL6A5643V,2026-06-10 11:50:58,B156HAN2U_VB,TFT,ARRAY,01145,AA1774,CAAOI300,CAPIC600,...,2026-06-08 23:19:07,2.0,1000.0,0.0,0.000000,[],NO_SAME_POINT,source found but no same point; source_defect_...,2026-06-11 16:22:45,2026-06-11 16:22:45
330,331,YL6A5643V,2026-06-10 11:50:58,B156HAN2U_VB,TFT,ARRAY,01145,AA1774,CAAOI300,CAPIC600,...,2026-06-07 10:25:37,1.0,1000.0,1.0,1.000000,"[{""cell"": {""cell_defect_uid"": ""YL6A5643V|2026-...",MATCHED,source_same_point_cnt=1; cell_same_point_cnt=2...,2026-06-11 16:22:45,2026-06-11 16:22:45
331,332,YL6A5643W,2026-06-10 11:56:07,B156HAN2U_VB,TFT,ARRAY,01145,AA1774,CAAOI300,CAPIC600,...,2026-06-06 15:27:00,260.0,1000.0,4.0,0.015385,"[{""cell"": {""cell_defect_uid"": ""YL6A5643W|2026-...",MATCHED,source_same_point_cnt=4; cell_same_point_cnt=4...,2026-06-11 16:22:45,2026-06-11 16:22:45
332,333,YL6A5643W,2026-06-10 11:56:07,B156HAN2U_VB,TFT,ARRAY,01145,AA1774,CAAOI300,CAPIC600,...,2026-06-08 23:19:07,5.0,1000.0,2.0,0.400000,"[{""cell"": {""cell_defect_uid"": ""YL6A5643W|2026-...",MATCHED,source_same_point_cnt=2; cell_same_point_cnt=2...,2026-06-11 16:22:45,2026-06-11 16:22:45


In [4]:
df = db.get_rows('incoming_glass_summary_202606', {'sheet_id': 'L86A5246J'})
df

[{'id': 211,
  'sheet_id': 'L86A5246J',
  'scan_time': datetime.datetime(2026, 6, 9, 16, 46, 3),
  'model_no': 'M215HAN1G_VB',
  'abbr_cat': 'TFT',
  'process': 'ARRAY',
  'recipe_id': '07086',
  'cassette_id': 'CA0335',
  'cell_aoi': 'CAAOI300',
  'cell_line_id': 'CAPIC200',
  'pi_time': datetime.datetime(2026, 6, 8, 5, 53, 9),
  'cell_op': 'API',
  'cell_defect_cnt': 210,
  'source_station_cnt': 3,
  'source_found_station_cnt': 1,
  'total_source_defect_cnt': 2,
  'total_same_point_defect_cnt': 36,
  'total_same_point_rate': 0.17142857142857143,
  'cf_oc_source_defect_cnt': 0,
  'cf_ps_source_defect_cnt': 0,
  'array_mor_source_defect_cnt': 0,
  'array_tar_source_defect_cnt': 0,
  'array_tos_source_defect_cnt': 2,
  'cf_oc_same_point_cnt': 0,
  'cf_ps_same_point_cnt': 0,
  'array_mor_same_point_cnt': 0,
  'array_tar_same_point_cnt': 0,
  'array_tos_same_point_cnt': 36,
  'judge': 'MATCHED',
  'judge_detail': 'cell_defect_cnt=210; source_found_station_cnt=1; total_source_defect_cnt=2;

In [ ]:
[{
    "cell": {
        "cell_defect_uid": "AE1UA600E0|2026-06-08 10:27:24|1123",
        "chip_id": "", 
        "defect_code": "", 
        "retype_def_code": "", 
        "defect_size": "S", 
        "ori_x": 592905.0,
        "ori_y": 899232.0, 
        "trans_x": 592905.0, 
        "trans_y": 600768.0, 
        "image_name": "", 
        "img_url_path": "http://l6apaimg103/dms/CELAIDI_L6A/PIT/2606/08/CAPIT203/AE1UA600E0/1027/"
        }, 
    "source": {
        "process": "CF", 
        "source_op_id": "OC", 
        "scan_time": "2026-06-07 01:52:29", 
        "chip_id": "AE1UA600E0K",
        "defect_no": "0007", 
        "defect_code": "TB", 
        "defect_size": "M", 
        "ori_x": 898215.0, 
        "ori_y": 591007.0, 
        "trans_x": 591007.0, 
        "trans_y": 601785.0, 
        "image_name": "RAE1UA600E0_0007_001_898215_591007.jpg", 
        "img_url_path": "http://tcweb002.corpnet.auo.com/fafle001/image/OC/0/AE1UA600E0/RAE1UA600E0_0007_001_898215_591007.jpg", 
        "source_group_key": "CF|AE1UA600E0|OC|2026-06-07 01:52:29"
        }, 
    "match": {
        "offset": 3000.0, 
        "dx": 1898.0, 
        "dy": 1017.0, 
        "distance": 2153.2981679275167, 
        "rank": 1, 
        "is_nearest": 1
        }
    }]